## Introducción

La arquitectura U-Net es uno de los modelos más utilizados en segmentación biomédica, principalmente por su capacidad para combinar información global y local mediante un camino de codificación y otro de decodificación unidos por *skip connections*. Sin embargo, el rendimiento final del modelo puede variar enormemente según cómo se configuren sus parámetros estructurales y de entrenamiento. Por ello, en este trabajo realizamos un análisis sistemático del comportamiento del modelo al modificar cada parámetro de forma independiente, con el objetivo de entender cómo afectan a la estabilidad del aprendizaje, a las curvas de pérdida y exactitud, y al rendimiento general.

En total, se evaluaron **14 parámetros**, tanto arquitectónicos como de entrenamiento, que fueron:

1. **skip_merge**: add, concatenate, attention  
2. **loss_cfg**: BCE, Dice, BCE + Dice  
3. **upsampling_method**: transposed conv, bilinear, nearest  
4. **lr_schedule**: constante, cosine decay, reduce-on-plateau  
5. **batch_shuffle**: (bs2, sh8), (bs4, sh32), (bs8, sh128)  
6. **optimizer_cfg**: Adam, AdamW, SGD + momentum  
7. **aug_profile**: none, geométrico, geom+fotométrico  
8. **down_op**: MaxPool, AvgPool, Strided Conv  
9. **dropout + L2**: (0.0, 0), (0.2, 1e-5), (0.4, 1e-4)  
10. **norm_layer**: BatchNorm, GroupNorm, None  
11. **kernel_size**: 3, 5, 7  
12. **width_multiplier**: 32, 64, 96  
13. **levels (profundidad de la UNet)**: 3, 4, 5  
14. **block_type**: simple conv block, residual block, depthwise-separable block  

La idea principal fue estudiar cómo cada uno de estos parámetros influye en el comportamiento del modelo, especialmente considerando que partimos de una U-Net relativamente pequeña y con capacidad limitada. Esto hace que el modelo pueda aprender demasiado rápido, sobreajustarse o incluso volverse inestable dependiendo de la configuración, algo que se observa en varios de los resultados obtenidos.

El análisis de estos 14 parámetros nos permite identificar configuraciones más adecuadas, comprender por qué algunas combinaciones fallan y establecer una base para optimizar modelos de segmentación biomédica en futuros trabajos.


## **Parámetro de SKIP MERGE METHOD**

In [ ]:
# ============================
# CONFIGURACIÓN DEL EXPERIMENTO
# ============================

# Cambia SOLO esta línea entre ejecuciones:
#   "concatenate"  (baseline)
#   "add"
#   "attention"    (skip atenuado con attention gate simple)

SKIP_MERGE_METHOD = "concatenate"
# SKIP_MERGE_METHOD = "add"
# SKIP_MERGE_METHOD = "attention"


# ============================
# IMPORTS
# ============================
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.layers import (
    Dense, Conv2D, MaxPooling2D, Dropout, BatchNormalization,
    Conv2DTranspose, Activation, concatenate, Input
)
from tensorflow.keras import Model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

import matplotlib.pyplot as plt


# ============================
# RUTAS DEL DATASET EN KAGGLE
# ============================
# Dataset: nikitamanaenkov/fundus-image-dataset-for-vessel-segmentation
BASE_PATH = "/kaggle/input/fundus-image-dataset-for-vessel-segmentation"
TRAIN_PATH = os.path.join(BASE_PATH, "train")
TEST_PATH  = os.path.join(BASE_PATH, "test")


# ============================
# CARGA DE DATOS
# ============================
def load_data(path, img_size):
    images = []
    masks = []
    num_images = 100  # puedes reducir si quieres entrenar más rápido

    img_dir = os.path.join(path, "Original")
    mask_dir = os.path.join(path, "Ground truth")

    img_files = sorted(os.listdir(img_dir))[:num_images]
    mask_files = sorted(os.listdir(mask_dir))[:num_images]

    for img_file, mask_file in zip(img_files, mask_files):
        img_path = os.path.join(img_dir, img_file)
        mask_path = os.path.join(mask_dir, mask_file)

        img = load_img(img_path, target_size=img_size)
        mask = load_img(mask_path, target_size=img_size, color_mode="grayscale")

        img = img_to_array(img) / 255.0
        mask = img_to_array(mask) / 255.0

        images.append(img)
        masks.append(mask)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)


IMG_SIZE = (512, 512)
x_train, y_train = load_data(TRAIN_PATH, IMG_SIZE)
x_test,  y_test  = load_data(TEST_PATH,  IMG_SIZE)

print("Train shapes:", x_train.shape, y_train.shape)
print("Test  shapes:", x_test.shape,  y_test.shape)


# ============================
# DATASET TF + AUGMENTACIÓN
# ============================
def augment_image(image, mask):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    return image, mask


def tf_dataset(x, y, batch_size=8, augment=True):
    dataset = tf.data.Dataset.from_tensor_slices((x, y))
    dataset = dataset.shuffle(buffer_size=len(x))

    if augment:
        dataset = dataset.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)

    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


BATCH_SIZE = 8
train_dataset = tf_dataset(x_train, y_train, batch_size=BATCH_SIZE, augment=True)
test_dataset  = tf_dataset(x_test,  y_test,  batch_size=BATCH_SIZE, augment=False)


# ============================
# DEFINICIÓN DEL MODELO (UNET)
# ============================
def conv_block(input_tensor, filters):
    x = Conv2D(filters, 3, padding="same")(input_tensor)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = Conv2D(filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    return x


def encoder_block(input_tensor, filters):
    x = conv_block(input_tensor, filters)
    p = MaxPooling2D((2, 2))(x)
    return x, p


def attention_gate(g, x, filters):
    """
    Simple attention gate:
    g: features de la rama decodificadora (después de upsampling)
    x: features de la skip connection (rama codificadora)
    """
    g1 = Conv2D(filters, 1, padding="same")(g)
    x1 = Conv2D(filters, 1, padding="same")(x)
    psi = Activation("relu")(g1 + x1)
    psi = Conv2D(1, 1, padding="same", activation="sigmoid")(psi)
    # atenuamos la skip con el mapa de atención
    x_att = x * psi
    return x_att


def decoder_block(input_tensor, filters, skip_tensor):
    # Upsampling fijo: tu versión original con Conv2DTranspose
    x = Conv2DTranspose(filters, 2, strides=2, padding="same")(input_tensor)

    # FUSIÓN SEGÚN skip_merge
    if SKIP_MERGE_METHOD == "concatenate":
        merged = concatenate([x, skip_tensor])

    elif SKIP_MERGE_METHOD == "add":
        # Asumimos mismo tamaño de canales, que se cumple en UNet estándar
        merged = x + skip_tensor

    elif SKIP_MERGE_METHOD == "attention":
        skip_att = attention_gate(x, skip_tensor, filters)
        merged = concatenate([x, skip_att])

    else:
        raise ValueError(f"Unknown SKIP_MERGE_METHOD: {SKIP_MERGE_METHOD}")

    x = conv_block(merged, filters)
    return x


def build_model():
    input_layer = Input(shape=(512, 512, 3))

    x1, p1 = encoder_block(input_layer, 64)
    x2, p2 = encoder_block(p1, 128)
    x3, p3 = encoder_block(p2, 256)
    x4, p4 = encoder_block(p3, 512)

    a1 = conv_block(p4, 1024)

    d1 = decoder_block(a1, 512, x4)
    d2 = decoder_block(d1, 256, x3)
    d3 = decoder_block(d2, 128, x2)
    d4 = decoder_block(d3, 64,  x1)

    output_layer = Conv2D(1, (1, 1), padding="same", activation="sigmoid")(d4)

    model = Model(input_layer, output_layer)
    return model


# ============================
# COMPILAR Y ENTRENAR
# ============================
model = build_model()
model.summary()

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

checkpoint_path = f"/kaggle/working/best_model_skipmerge_{SKIP_MERGE_METHOD}.keras"

checkpoint = ModelCheckpoint(
    checkpoint_path,
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    mode='min',
    verbose=1,
    restore_best_weights=True
)

EPOCHS = 100  # ajústalo según el tiempo que tengas

history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=test_dataset,
    callbacks=[checkpoint, early_stopping],
    verbose=1
)


# ============================
# GUARDAR HISTÓRICO COMPLETO DE ÉPOCAS EN CSV
# ============================
history_dict = history.history
num_epochs_run = len(history_dict["loss"])

rows = []
for epoch_idx in range(num_epochs_run):
    row = {
        "parameter": "skip_merge",
        "value": SKIP_MERGE_METHOD,
        "epoch": epoch_idx + 1,
        "loss": history_dict["loss"][epoch_idx],
        "accuracy": history_dict["accuracy"][epoch_idx],
        "val_loss": history_dict["val_loss"][epoch_idx],
        "val_accuracy": history_dict["val_accuracy"][epoch_idx],
    }
    rows.append(row)

df_history = pd.DataFrame(rows)

history_csv_path = f"/kaggle/working/history_skipmerge_{SKIP_MERGE_METHOD}.csv"
df_history.to_csv(history_csv_path, index=False)

print("Histórico de épocas guardado en:", history_csv_path)
print(df_history.head())
print(df_history.tail())


2025-12-08 15:56:07.216610: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765209367.532715      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765209367.623239      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 512, 512,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 512, 512,  │      1,792 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 512, 512,  │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 512, 512,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 512, 512,  │     36,928 │ activation[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 512, 512,  │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 512, 512,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 256, 256,  │          0 │ activation_1[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 256, 256,  │     73,856 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        512 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 256, 256,  │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 256, 256,  │    147,584 │ activation_2[0][… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        512 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 256, 256,  │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 128, 128,  │          0 │ activation_3[0][… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 128, 128,  │    295,168 │ max_pooling2d_1[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │      1,024 │ conv2d_4[0][0]  

 Total params: 31,055,297 (118.47 MB)

 Trainable params: 31,043,521 (118.42 MB)

 Non-trainable params: 11,776 (46.00 KB)

Epoch 1/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 109s/step - accuracy: 0.6119 - loss: 0.6944  
Epoch 1: val_loss improved from inf to 86.18935, saving model to /kaggle/working/best_model_skipmerge_concatenate.keras
13/13 ━━━━━━━━━━━━━━━━━━━━ 1828s 141s/step - accuracy: 0.6247 - loss: 0.6806 - val_accuracy: 0.0699 - val_loss: 86.1894
Epoch 2/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 111s/step - accuracy: 0.9298 - loss: 0.2824  
Epoch 2: val_loss improved from 86.18935 to 17.80661, saving model to /kaggle/working/best_model_skipmerge_concatenate.keras
13/13 ━━━━━━━━━━━━━━━━━━━━ 1831s 143s/step - accuracy: 0.9298 - loss: 0.2820 - val_accuracy: 0.1334 - val_loss: 17.8066
Epoch 3/100
 1/13 ━━━━━━━━━━━━━━━━━━━━ 23:16 116s/step - accuracy: 0.9277 - loss: 0.2641

## **Parámetro de LR_SCHEDULE**

In [ ]:
# ============================
# CONFIGURACIÓN DEL EXPERIMENTO
# ============================

# Cambia SOLO esta línea entre ejecuciones:
#   "const"
#   "reduce_on_plateau"
#   "cosine_decay"

# LR_SCHEDULE = "const"
# LR_SCHEDULE = "reduce_on_plateau"
LR_SCHEDULE = "cosine_decay"

BASE_LR = 1e-3  # learning rate base


# ============================
# IMPORTS
# ============================
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.layers import (
    Dense, Conv2D, MaxPooling2D, Dropout, BatchNormalization,
    Conv2DTranspose, Activation, concatenate, Input
)
from tensorflow.keras import Model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

import matplotlib.pyplot as plt


# ============================
# RUTAS DEL DATASET EN KAGGLE
# ============================
# Dataset: nikitamanaenkov/fundus-image-dataset-for-vessel-segmentation
BASE_PATH = "/kaggle/input/fundus-image-dataset-for-vessel-segmentation"
TRAIN_PATH = os.path.join(BASE_PATH, "train")
TEST_PATH  = os.path.join(BASE_PATH, "test")


# ============================
# CARGA DE DATOS
# ============================
def load_data(path, img_size):
    images = []
    masks = []
    num_images = 100  # puedes reducirlo si quieres ir más rápido

    img_dir = os.path.join(path, "Original")
    mask_dir = os.path.join(path, "Ground truth")

    img_files = sorted(os.listdir(img_dir))[:num_images]
    mask_files = sorted(os.listdir(mask_dir))[:num_images]

    for img_file, mask_file in zip(img_files, mask_files):
        img_path = os.path.join(img_dir, img_file)
        mask_path = os.path.join(mask_dir, mask_file)

        img = load_img(img_path, target_size=img_size)
        mask = load_img(mask_path, target_size=img_size, color_mode="grayscale")

        img = img_to_array(img) / 255.0
        mask = img_to_array(mask) / 255.0

        images.append(img)
        masks.append(mask)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)


IMG_SIZE = (512, 512)
x_train, y_train = load_data(TRAIN_PATH, IMG_SIZE)
x_test,  y_test  = load_data(TEST_PATH,  IMG_SIZE)

print("Train shapes:", x_train.shape, y_train.shape)
print("Test  shapes:", x_test.shape,  y_test.shape)


# ============================
# DATASET TF + AUGMENTACIÓN
# ============================
def augment_image(image, mask):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    return image, mask


def tf_dataset(x, y, batch_size=8, augment=True):
    dataset = tf.data.Dataset.from_tensor_slices((x, y))
    dataset = dataset.shuffle(buffer_size=len(x))

    if augment:
        dataset = dataset.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)

    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


BATCH_SIZE = 8
train_dataset = tf_dataset(x_train, y_train, batch_size=BATCH_SIZE, augment=True)
test_dataset  = tf_dataset(x_test,  y_test,  batch_size=BATCH_SIZE, augment=False)

steps_per_epoch = int(np.ceil(len(x_train) / BATCH_SIZE))


# ============================
# DEFINICIÓN DEL MODELO (UNET BASE)
# ============================
def conv_block(input_tensor, filters):
    x = Conv2D(filters, 3, padding="same")(input_tensor)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = Conv2D(filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    return x


def encoder_block(input_tensor, filters):
    x = conv_block(input_tensor, filters)
    p = MaxPooling2D((2, 2))(x)
    return x, p


def decoder_block(input_tensor, filters, skip_tensor):
    # Upsampling original con Conv2DTranspose
    x = Conv2DTranspose(filters, 2, strides=2, padding="same")(input_tensor)
    x = concatenate([x, skip_tensor])
    x = conv_block(x, filters)
    return x


def build_model():
    input_layer = Input(shape=(512, 512, 3))

    x1, p1 = encoder_block(input_layer, 64)
    x2, p2 = encoder_block(p1, 128)
    x3, p3 = encoder_block(p2, 256)
    x4, p4 = encoder_block(p3, 512)

    a1 = conv_block(p4, 1024)

    d1 = decoder_block(a1, 512, x4)
    d2 = decoder_block(d1, 256, x3)
    d3 = decoder_block(d2, 128, x2)
    d4 = decoder_block(d3, 64,  x1)

    output_layer = Conv2D(1, (1, 1), padding="same", activation="sigmoid")(d4)

    model = Model(input_layer, output_layer)
    return model


# ============================
# OPTIMIZER + CALLBACKS SEGÚN lr_schedule
# ============================
def get_optimizer_and_callbacks(lr_schedule_name, base_lr, checkpoint_path):
    extra_callbacks = []

    if lr_schedule_name == "const":
        optimizer = tf.keras.optimizers.Adam(learning_rate=base_lr)

    elif lr_schedule_name == "reduce_on_plateau":
        optimizer = tf.keras.optimizers.Adam(learning_rate=base_lr)
        lr_cb = ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1
        )
        extra_callbacks.append(lr_cb)

    elif lr_schedule_name == "cosine_decay":
        decay_steps = steps_per_epoch * 100  # aprox: EPOCHS * steps_per_epoch (ajustable)
        lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
            initial_learning_rate=base_lr,
            decay_steps=decay_steps,
            alpha=0.0
        )
        optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

    else:
        raise ValueError(f"Unknown LR_SCHEDULE: {lr_schedule_name}")

    # Callbacks comunes
    checkpoint = ModelCheckpoint(
        checkpoint_path,
        monitor='val_loss',
        save_best_only=True,
        mode='min',
        verbose=1
    )

    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=15,
        mode='min',
        verbose=1,
        restore_best_weights=True
    )

    callbacks = [checkpoint, early_stopping] + extra_callbacks
    return optimizer, callbacks


# ============================
# COMPILAR Y ENTRENAR
# ============================
checkpoint_path = f"/kaggle/working/best_model_lrschedule_{LR_SCHEDULE}.keras"

model = build_model()
model.summary()

optimizer, callbacks = get_optimizer_and_callbacks(LR_SCHEDULE, BASE_LR, checkpoint_path)

model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"])

EPOCHS = 100  # ajusta si quieres menos

history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=test_dataset,
    callbacks=callbacks,
    verbose=1
)


# ============================
# GUARDAR HISTÓRICO COMPLETO DE ÉPOCAS EN CSV
# ============================
history_dict = history.history
num_epochs_run = len(history_dict["loss"])

rows = []
for epoch_idx in range(num_epochs_run):
    row = {
        "parameter": "lr_schedule",
        "value": LR_SCHEDULE,
        "epoch": epoch_idx + 1,
        "loss": history_dict["loss"][epoch_idx],
        "accuracy": history_dict["accuracy"][epoch_idx],
        "val_loss": history_dict["val_loss"][epoch_idx],
        "val_accuracy": history_dict["val_accuracy"][epoch_idx],
    }
    rows.append(row)

df_history = pd.DataFrame(rows)

history_csv_path = f"/kaggle/working/history_lrschedule_{LR_SCHEDULE}.csv"
df_history.to_csv(history_csv_path, index=False)

print("Histórico de épocas guardado en:", history_csv_path)
print(df_history.head())
print(df_history.tail())


## **Parámetro de UPSAMPLING METHOD**

In [ ]:
# ============================
# CONFIGURACIÓN DEL EXPERIMENTO
# ============================

# Cambia SOLO esta línea entre ejecuciones:
#   "transposed"       (baseline, tu modelo original)
#   "upsample_nearest" (UpSampling2D nearest + Conv2D)
#   "upsample_bilinear" (UpSampling2D bilinear + Conv2D)

# UPSAMPLING_METHOD = "transposed"
# UPSAMPLING_METHOD = "upsample_nearest"
UPSAMPLING_METHOD = "upsample_bilinear"

CSV_PATH = "/kaggle/working/experiments_results_unet.csv"


# ============================
# IMPORTS
# ============================
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.layers import (
    Dense, Conv2D, MaxPooling2D, Dropout, BatchNormalization,
    Conv2DTranspose, Activation, concatenate, Input, UpSampling2D
)
from tensorflow.keras import Model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

import matplotlib.pyplot as plt


# ============================
# RUTAS DEL DATASET EN KAGGLE
# ============================
# Dataset: nikitamanaenkov/fundus-image-dataset-for-vessel-segmentation
BASE_PATH = "/kaggle/input/fundus-image-dataset-for-vessel-segmentation"
TRAIN_PATH = os.path.join(BASE_PATH, "train")
TEST_PATH  = os.path.join(BASE_PATH, "test")


# ============================
# CARGA DE DATOS
# ============================
def load_data(path, img_size):
    images = []
    masks = []
    num_images = 100  # puedes reducir si quieres ir más rápido

    img_dir = os.path.join(path, "Original")
    mask_dir = os.path.join(path, "Ground truth")

    img_files = sorted(os.listdir(img_dir))[:num_images]
    mask_files = sorted(os.listdir(mask_dir))[:num_images]

    for img_file, mask_file in zip(img_files, mask_files):
        img_path = os.path.join(img_dir, img_file)
        mask_path = os.path.join(mask_dir, mask_file)

        img = load_img(img_path, target_size=img_size)
        mask = load_img(mask_path, target_size=img_size, color_mode="grayscale")

        img = img_to_array(img) / 255.0
        mask = img_to_array(mask) / 255.0

        images.append(img)
        masks.append(mask)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)


IMG_SIZE = (512, 512)
x_train, y_train = load_data(TRAIN_PATH, IMG_SIZE)
x_test,  y_test  = load_data(TEST_PATH, IMG_SIZE)

print("Train shapes:", x_train.shape, y_train.shape)
print("Test  shapes:", x_test.shape,  y_test.shape)


# ============================
# DATASET TF + AUGMENTACIÓN
# ============================
def augment_image(image, mask):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    return image, mask


def tf_dataset(x, y, batch_size=8, augment=True):
    dataset = tf.data.Dataset.from_tensor_slices((x, y))
    dataset = dataset.shuffle(buffer_size=len(x))

    if augment:
        dataset = dataset.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)

    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


BATCH_SIZE = 8
train_dataset = tf_dataset(x_train, y_train, batch_size=BATCH_SIZE, augment=True)
test_dataset  = tf_dataset(x_test,  y_test,  batch_size=BATCH_SIZE, augment=False)


# ============================
# DEFINICIÓN DEL MODELO (TU UNET)
# ============================
def conv_block(input_tensor, filters):
    x = Conv2D(filters, 3, padding="same")(input_tensor)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = Conv2D(filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    return x


def encoder_block(input_tensor, filters):
    x = conv_block(input_tensor, filters)
    p = MaxPooling2D((2, 2))(x)
    return x, p


def decoder_block(input_tensor, filters, skip_tensor):
    """
    MISMA FUNCIÓN PARA TODOS LOS EXPERIMENTOS.
    SOLO CAMBIA EL COMPORTAMIENTO SEGÚN UPSAMPLING_METHOD.
    """

    if UPSAMPLING_METHOD == "transposed":
        # Tu versión original:
        x = Conv2DTranspose(filters, 2, strides=2, padding="same")(input_tensor)

    elif UPSAMPLING_METHOD == "upsample_nearest":
        # Upsampling vecino más cercano + Conv2D:
        x = UpSampling2D(size=(2, 2), interpolation="nearest")(input_tensor)
        x = Conv2D(filters, 2, padding="same")(x)

    elif UPSAMPLING_METHOD == "upsample_bilinear":
        # Upsampling bilinear + Conv2D:
        x = UpSampling2D(size=(2, 2), interpolation="bilinear")(input_tensor)
        x = Conv2D(filters, 2, padding="same")(x)

    else:
        raise ValueError(f"Unknown UPSAMPLING_METHOD: {UPSAMPLING_METHOD}")

    x = concatenate([x, skip_tensor])
    x = conv_block(x, filters)
    return x


def build_model():
    input_layer = Input(shape=(512, 512, 3))

    x1, p1 = encoder_block(input_layer, 64)
    x2, p2 = encoder_block(p1, 128)
    x3, p3 = encoder_block(p2, 256)
    x4, p4 = encoder_block(p3, 512)

    a1 = conv_block(p4, 1024)

    d1 = decoder_block(a1, 512, x4)
    d2 = decoder_block(d1, 256, x3)
    d3 = decoder_block(d2, 128, x2)
    d4 = decoder_block(d3, 64,  x1)

    output_layer = Conv2D(1, (1, 1), padding="same", activation="sigmoid")(d4)

    model = Model(input_layer, output_layer)
    return model


# ============================
# COMPILAR Y ENTRENAR
# ============================
model = build_model()
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

checkpoint_path = f"/kaggle/working/best_model_{UPSAMPLING_METHOD}.keras"

checkpoint = ModelCheckpoint(
    checkpoint_path,
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    mode='min',
    verbose=1,
    restore_best_weights=True
)

EPOCHS = 100  # puedes bajar a 30–50 si va lento

history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=test_dataset,
    callbacks=[checkpoint, early_stopping],
    verbose=1
)


# ============================
# GUARDAR HISTÓRICO COMPLETO DE ÉPOCAS EN CSV
# ============================

history_dict = history.history
num_epochs_run = len(history_dict["loss"])

rows = []
for epoch_idx in range(num_epochs_run):
    row = {
        "parameter": "upsampling_method",
        "value": UPSAMPLING_METHOD,
        "epoch": epoch_idx + 1,  # empezamos en 1 para humanos
        "loss": history_dict["loss"][epoch_idx],
        "accuracy": history_dict["accuracy"][epoch_idx],
        "val_loss": history_dict["val_loss"][epoch_idx],
        "val_accuracy": history_dict["val_accuracy"][epoch_idx],
    }
    rows.append(row)

df_history = pd.DataFrame(rows)

# Un CSV distinto por cada método, en /kaggle/working
history_csv_path = f"/kaggle/working/history_upsampling_{UPSAMPLING_METHOD}.csv"
df_history.to_csv(history_csv_path, index=False)

print("Histórico de épocas guardado en:", history_csv_path)
print(df_history.head())
print(df_history.tail())


## **Parámetro de LOSS CFG**

In [ ]:
# ============================
# CONFIGURACIÓN DEL EXPERIMENTO
# ============================

# Cambia SOLO esta línea entre ejecuciones:
#   "BCE"
#   "Dice"
#   "BCE_plus_Dice"   (BCE + 1·Dice)

# LOSS_CFG = "BCE"
# LOSS_CFG = "Dice"
LOSS_CFG = "BCE_plus_Dice"


# ============================
# IMPORTS
# ============================
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.layers import (
    Dense, Conv2D, MaxPooling2D, Dropout, BatchNormalization,
    Conv2DTranspose, Activation, concatenate, Input
)
from tensorflow.keras import Model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

import matplotlib.pyplot as plt


# ============================
# RUTAS DEL DATASET EN KAGGLE
# ============================
# Dataset: nikitamanaenkov/fundus-image-dataset-for-vessel-segmentation
BASE_PATH = "/kaggle/input/fundus-image-dataset-for-vessel-segmentation"
TRAIN_PATH = os.path.join(BASE_PATH, "train")
TEST_PATH  = os.path.join(BASE_PATH, "test")


# ============================
# CARGA DE DATOS
# ============================
def load_data(path, img_size):
    images = []
    masks = []
    num_images = 100  # puedes reducirlo si quieres entrenar más rápido

    img_dir = os.path.join(path, "Original")
    mask_dir = os.path.join(path, "Ground truth")

    img_files = sorted(os.listdir(img_dir))[:num_images]
    mask_files = sorted(os.listdir(mask_dir))[:num_images]

    for img_file, mask_file in zip(img_files, mask_files):
        img_path = os.path.join(img_dir, img_file)
        mask_path = os.path.join(mask_dir, mask_file)

        img = load_img(img_path, target_size=img_size)
        mask = load_img(mask_path, target_size=img_size, color_mode="grayscale")

        img = img_to_array(img) / 255.0
        mask = img_to_array(mask) / 255.0

        images.append(img)
        masks.append(mask)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)


IMG_SIZE = (512, 512)
x_train, y_train = load_data(TRAIN_PATH, IMG_SIZE)
x_test,  y_test  = load_data(TEST_PATH,  IMG_SIZE)

print("Train shapes:", x_train.shape, y_train.shape)
print("Test  shapes:", x_test.shape,  y_test.shape)


# ============================
# DATASET TF + AUGMENTACIÓN
# ============================
def augment_image(image, mask):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    return image, mask


def tf_dataset(x, y, batch_size=8, augment=True):
    dataset = tf.data.Dataset.from_tensor_slices((x, y))
    dataset = dataset.shuffle(buffer_size=len(x))

    if augment:
        dataset = dataset.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)

    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


BATCH_SIZE = 8
train_dataset = tf_dataset(x_train, y_train, batch_size=BATCH_SIZE, augment=True)
test_dataset  = tf_dataset(x_test,  y_test,  batch_size=BATCH_SIZE, augment=False)


# ============================
# FUNCIÓN DE PÉRDIDA SEGÚN loss_cfg
# ============================
def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    numerator = 2.0 * intersection + smooth
    denominator = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth
    dice_coef = numerator / denominator
    return 1.0 - dice_coef  # loss = 1 - Dice


bce_loss_fn = tf.keras.losses.BinaryCrossentropy()

def bce_plus_dice_loss(y_true, y_pred):
    return bce_loss_fn(y_true, y_pred) + dice_loss(y_true, y_pred)  # α = 1


def get_loss_fn(loss_cfg):
    if loss_cfg == "BCE":
        return bce_loss_fn
    elif loss_cfg == "Dice":
        return dice_loss
    elif loss_cfg == "BCE_plus_Dice":
        return bce_plus_dice_loss
    else:
        raise ValueError(f"Unknown LOSS_CFG: {loss_cfg}")


# ============================
# DEFINICIÓN DEL MODELO (UNET BASE)
# ============================
def conv_block(input_tensor, filters):
    x = Conv2D(filters, 3, padding="same")(input_tensor)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = Conv2D(filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    return x


def encoder_block(input_tensor, filters):
    x = conv_block(input_tensor, filters)
    p = MaxPooling2D((2, 2))(x)
    return x, p


def decoder_block(input_tensor, filters, skip_tensor):
    # Upsampling original con Conv2DTranspose
    x = Conv2DTranspose(filters, 2, strides=2, padding="same")(input_tensor)
    x = concatenate([x, skip_tensor])
    x = conv_block(x, filters)
    return x


def build_model():
    input_layer = Input(shape=(512, 512, 3))

    x1, p1 = encoder_block(input_layer, 64)
    x2, p2 = encoder_block(p1, 128)
    x3, p3 = encoder_block(p2, 256)
    x4, p4 = encoder_block(p3, 512)

    a1 = conv_block(p4, 1024)

    d1 = decoder_block(a1, 512, x4)
    d2 = decoder_block(d1, 256, x3)
    d3 = decoder_block(d2, 128, x2)
    d4 = decoder_block(d3, 64,  x1)

    output_layer = Conv2D(1, (1, 1), padding="same", activation="sigmoid")(d4)

    model = Model(input_layer, output_layer)
    return model


# ============================
# COMPILAR Y ENTRENAR
# ============================
loss_fn = get_loss_fn(LOSS_CFG)

model = build_model()
model.summary()

model.compile(optimizer="adam", loss=loss_fn, metrics=["accuracy"])

checkpoint_path = f"/kaggle/working/best_model_losscfg_{LOSS_CFG}.keras"

checkpoint = ModelCheckpoint(
    checkpoint_path,
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    mode='min',
    verbose=1,
    restore_best_weights=True
)

EPOCHS = 100  # ajusta según el tiempo que tengas

history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=test_dataset,
    callbacks=[checkpoint, early_stopping],
    verbose=1
)


# ============================
# GUARDAR HISTÓRICO COMPLETO DE ÉPOCAS EN CSV
# ============================
history_dict = history.history
num_epochs_run = len(history_dict["loss"])

rows = []
for epoch_idx in range(num_epochs_run):
    row = {
        "parameter": "loss_cfg",
        "value": LOSS_CFG,
        "epoch": epoch_idx + 1,
        "loss": history_dict["loss"][epoch_idx],
        "accuracy": history_dict["accuracy"][epoch_idx],
        "val_loss": history_dict["val_loss"][epoch_idx],
        "val_accuracy": history_dict["val_accuracy"][epoch_idx],
    }
    rows.append(row)

df_history = pd.DataFrame(rows)

history_csv_path = f"/kaggle/working/history_losscfg_{LOSS_CFG}.csv"
df_history.to_csv(history_csv_path, index=False)

print("Histórico de épocas guardado en:", history_csv_path)
print(df_history.head())
print(df_history.tail())


## **Parámetro de BATCH + SHUFFLE** 

In [ ]:
# ============================
# CONFIGURACIÓN DEL EXPERIMENTO
# ============================

# Cambia SOLO esta línea entre ejecuciones:
#   "bs2_sh8"    -> batch_size=2,  shuffle_buffer=8
#   "bs4_sh32"   -> batch_size=4,  shuffle_buffer=32
#   "bs8_sh128"  -> batch_size=8,  shuffle_buffer=128

# BATCH_SHUFFLE_CFG = "bs2_sh8"
# BATCH_SHUFFLE_CFG = "bs4_sh32"
BATCH_SHUFFLE_CFG = "bs8_sh128"


# ============================
# IMPORTS
# ============================
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.layers import (
    Dense, Conv2D, MaxPooling2D, Dropout, BatchNormalization,
    Conv2DTranspose, Activation, concatenate, Input
)
from tensorflow.keras import Model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

import matplotlib.pyplot as plt


# ============================
# MAPEO DE CONFIG -> batch + shuffle
# ============================
def get_batch_and_shuffle(cfg_name):
    if cfg_name == "bs2_sh8":
        return 2, 8
    elif cfg_name == "bs4_sh32":
        return 4, 32
    elif cfg_name == "bs8_sh128":
        return 8, 128
    else:
        raise ValueError(f"Unknown BATCH_SHUFFLE_CFG: {cfg_name}")


BATCH_SIZE, SHUFFLE_BUFFER = get_batch_and_shuffle(BATCH_SHUFFLE_CFG)
print(f"Using config {BATCH_SHUFFLE_CFG}: batch_size={BATCH_SIZE}, shuffle_buffer={SHUFFLE_BUFFER}")


# ============================
# RUTAS DEL DATASET EN KAGGLE
# ============================
# Dataset: nikitamanaenkov/fundus-image-dataset-for-vessel-segmentation
BASE_PATH = "/kaggle/input/fundus-image-dataset-for-vessel-segmentation"
TRAIN_PATH = os.path.join(BASE_PATH, "train")
TEST_PATH  = os.path.join(BASE_PATH, "test")


# ============================
# CARGA DE DATOS
# ============================
def load_data(path, img_size):
    images = []
    masks = []
    num_images = 100  # puedes reducirlo si quieres entrenar más rápido

    img_dir = os.path.join(path, "Original")
    mask_dir = os.path.join(path, "Ground truth")

    img_files = sorted(os.listdir(img_dir))[:num_images]
    mask_files = sorted(os.listdir(mask_dir))[:num_images]

    for img_file, mask_file in zip(img_files, mask_files):
        img_path = os.path.join(img_dir, img_file)
        mask_path = os.path.join(mask_dir, mask_file)

        img = load_img(img_path, target_size=img_size)
        mask = load_img(mask_path, target_size=img_size, color_mode="grayscale")

        img = img_to_array(img) / 255.0
        mask = img_to_array(mask) / 255.0

        images.append(img)
        masks.append(mask)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)


IMG_SIZE = (512, 512)
x_train, y_train = load_data(TRAIN_PATH, IMG_SIZE)
x_test,  y_test  = load_data(TEST_PATH,  IMG_SIZE)

print("Train shapes:", x_train.shape, y_train.shape)
print("Test  shapes:", x_test.shape,  y_test.shape)


# ============================
# DATASET TF + AUGMENTACIÓN
# ============================
def augment_image(image, mask):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    return image, mask


def tf_dataset(x, y, batch_size, shuffle_buffer, augment=True):
    dataset = tf.data.Dataset.from_tensor_slices((x, y))
    dataset = dataset.shuffle(buffer_size=shuffle_buffer)

    if augment:
        dataset = dataset.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)

    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


train_dataset = tf_dataset(x_train, y_train, batch_size=BATCH_SIZE, shuffle_buffer=SHUFFLE_BUFFER, augment=True)
test_dataset  = tf_dataset(x_test,  y_test,  batch_size=BATCH_SIZE, shuffle_buffer=SHUFFLE_BUFFER, augment=False)


# ============================
# DEFINICIÓN DEL MODELO (UNET BASE)
# ============================
def conv_block(input_tensor, filters):
    x = Conv2D(filters, 3, padding="same")(input_tensor)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = Conv2D(filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    return x


def encoder_block(input_tensor, filters):
    x = conv_block(input_tensor, filters)
    p = MaxPooling2D((2, 2))(x)
    return x, p


def decoder_block(input_tensor, filters, skip_tensor):
    x = Conv2DTranspose(filters, 2, strides=2, padding="same")(input_tensor)
    x = concatenate([x, skip_tensor])
    x = conv_block(x, filters)
    return x


def build_model():
    input_layer = Input(shape=(512, 512, 3))

    x1, p1 = encoder_block(input_layer, 64)
    x2, p2 = encoder_block(p1, 128)
    x3, p3 = encoder_block(p2, 256)
    x4, p4 = encoder_block(p3, 512)

    a1 = conv_block(p4, 1024)

    d1 = decoder_block(a1, 512, x4)
    d2 = decoder_block(d1, 256, x3)
    d3 = decoder_block(d2, 128, x2)
    d4 = decoder_block(d3, 64,  x1)

    output_layer = Conv2D(1, (1, 1), padding="same", activation="sigmoid")(d4)

    model = Model(input_layer, output_layer)
    return model


# ============================
# COMPILAR Y ENTRENAR
# ============================
model = build_model()
model.summary()

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

checkpoint_path = f"/kaggle/working/best_model_batchshuffle_{BATCH_SHUFFLE_CFG}.keras"

checkpoint = ModelCheckpoint(
    checkpoint_path,
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    mode='min',
    verbose=1,
    restore_best_weights=True
)

EPOCHS = 100  # ajusta si quieres menos

history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=test_dataset,
    callbacks=[checkpoint, early_stopping],
    verbose=1
)


# ============================
# GUARDAR HISTÓRICO COMPLETO DE ÉPOCAS EN CSV
# ============================
history_dict = history.history
num_epochs_run = len(history_dict["loss"])

rows = []
for epoch_idx in range(num_epochs_run):
    row = {
        "parameter": "batch_shuffle",
        "value": BATCH_SHUFFLE_CFG,
        "batch_size": BATCH_SIZE,
        "shuffle_buffer": SHUFFLE_BUFFER,
        "epoch": epoch_idx + 1,
        "loss": history_dict["loss"][epoch_idx],
        "accuracy": history_dict["accuracy"][epoch_idx],
        "val_loss": history_dict["val_loss"][epoch_idx],
        "val_accuracy": history_dict["val_accuracy"][epoch_idx],
    }
    rows.append(row)

df_history = pd.DataFrame(rows)

history_csv_path = f"/kaggle/working/history_batchshuffle_{BATCH_SHUFFLE_CFG}.csv"
df_history.to_csv(history_csv_path, index=False)

print("Histórico de épocas guardado en:", history_csv_path)
print(df_history.head())
print(df_history.tail())

## **Parámetro de AUG_PROFILE**

In [ ]:
# ============================
# CONFIGURACIÓN DEL EXPERIMENTO
# ============================

# Cambia SOLO esta línea entre ejecuciones:
#   "none"              -> sin augmentación
#   "geom"              -> flips + rotaciones
#   "geom_photometric"  -> geom + color jitter + blur

# AUG_PROFILE = "none"
# AUG_PROFILE = "geom"
AUG_PROFILE = "geom_photometric"


# ============================
# IMPORTS
# ============================
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.layers import (
    Dense, Conv2D, MaxPooling2D, Dropout, BatchNormalization,
    Conv2DTranspose, Activation, concatenate, Input
)
from tensorflow.keras import Model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

import matplotlib.pyplot as plt

print(f"Using AUG_PROFILE = {AUG_PROFILE}")


# ============================
# RUTAS DEL DATASET EN KAGGLE
# ============================
# Dataset: nikitamanaenkov/fundus-image-dataset-for-vessel-segmentation
BASE_PATH = "/kaggle/input/fundus-image-dataset-for-vessel-segmentation"
TRAIN_PATH = os.path.join(BASE_PATH, "train")
TEST_PATH  = os.path.join(BASE_PATH, "test")


# ============================
# CARGA DE DATOS
# ============================
def load_data(path, img_size):
    images = []
    masks = []
    num_images = 100  # puedes reducirlo si quieres ir más rápido

    img_dir = os.path.join(path, "Original")
    mask_dir = os.path.join(path, "Ground truth")

    img_files = sorted(os.listdir(img_dir))[:num_images]
    mask_files = sorted(os.listdir(mask_dir))[:num_images]

    for img_file, mask_file in zip(img_files, mask_files):
        img_path = os.path.join(img_dir, img_file)
        mask_path = os.path.join(mask_dir, mask_file)

        img = load_img(img_path, target_size=img_size)
        mask = load_img(mask_path, target_size=img_size, color_mode="grayscale")

        img = img_to_array(img) / 255.0
        mask = img_to_array(mask) / 255.0

        images.append(img)
        masks.append(mask)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)


IMG_SIZE = (512, 512)
x_train, y_train = load_data(TRAIN_PATH, IMG_SIZE)
x_test,  y_test  = load_data(TEST_PATH,  IMG_SIZE)

print("Train shapes:", x_train.shape, y_train.shape)
print("Test  shapes:", x_test.shape,  y_test.shape)


# ============================
# DATASET TF + AUGMENTACIÓN
# ============================

def random_geometric_augment(image, mask):
    # flip horizontal
    rnd = tf.random.uniform([])
    image = tf.cond(rnd > 0.5, lambda: tf.image.flip_left_right(image), lambda: image)
    mask  = tf.cond(rnd > 0.5, lambda: tf.image.flip_left_right(mask),  lambda: mask)

    # flip vertical
    rnd = tf.random.uniform([])
    image = tf.cond(rnd > 0.5, lambda: tf.image.flip_up_down(image), lambda: image)
    mask  = tf.cond(rnd > 0.5, lambda: tf.image.flip_up_down(mask),  lambda: mask)

    # rotación 0, 90, 180, 270 grados
    k = tf.random.uniform([], minval=0, maxval=4, dtype=tf.int32)
    image = tf.image.rot90(image, k)
    mask  = tf.image.rot90(mask,  k)

    return image, mask


def random_photometric_augment(image):
    # brillo y contraste (color jitter simple)
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)

    # blur sencillo con filtro caja 3x3
    kernel = tf.constant([[1., 1., 1.],
                          [1., 1., 1.],
                          [1., 1., 1.]], dtype=tf.float32)
    kernel = kernel / 9.0
    # convertimos a filtro depthwise para 3 canales
    kernel = tf.reshape(kernel, (3, 3, 1, 1))
    kernel = tf.tile(kernel, [1, 1, 3, 1])  # 3 canales

    image_4d = tf.expand_dims(image, axis=0)  # [1, H, W, 3]
    image_blur = tf.nn.depthwise_conv2d(
        image_4d, kernel, strides=[1, 1, 1, 1], padding="SAME"
    )
    image_blur = tf.squeeze(image_blur, axis=0)

    return image_blur


def augment_image(image, mask):
    # según perfil de augmentación
    if AUG_PROFILE == "none":
        return image, mask

    # siempre geométricos para "geom" y "geom_photometric"
    if AUG_PROFILE in ["geom", "geom_photometric"]:
        image, mask = random_geometric_augment(image, mask)

    # solo en el perfil más agresivo añadimos fotométricos
    if AUG_PROFILE == "geom_photometric":
        image = random_photometric_augment(image)

    return image, mask


def tf_dataset(x, y, batch_size=8, augment=True):
    dataset = tf.data.Dataset.from_tensor_slices((x, y))
    dataset = dataset.shuffle(buffer_size=len(x))

    if augment:
        dataset = dataset.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)

    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


BATCH_SIZE = 8
train_dataset = tf_dataset(x_train, y_train, batch_size=BATCH_SIZE, augment=True)
test_dataset  = tf_dataset(x_test,  y_test,  batch_size=BATCH_SIZE, augment=False)


# ============================
# DEFINICIÓN DEL MODELO (UNET BASE)
# ============================
def conv_block(input_tensor, filters):
    x = Conv2D(filters, 3, padding="same")(input_tensor)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = Conv2D(filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    return x


def encoder_block(input_tensor, filters):
    x = conv_block(input_tensor, filters)
    p = MaxPooling2D((2, 2))(x)
    return x, p


def decoder_block(input_tensor, filters, skip_tensor):
    x = Conv2DTranspose(filters, 2, strides=2, padding="same")(input_tensor)
    x = concatenate([x, skip_tensor])
    x = conv_block(x, filters)
    return x


def build_model():
    input_layer = Input(shape=(512, 512, 3))

    x1, p1 = encoder_block(input_layer, 64)
    x2, p2 = encoder_block(p1, 128)
    x3, p3 = encoder_block(p2, 256)
    x4, p4 = encoder_block(p3, 512)

    a1 = conv_block(p4, 1024)

    d1 = decoder_block(a1, 512, x4)
    d2 = decoder_block(d1, 256, x3)
    d3 = decoder_block(d2, 128, x2)
    d4 = decoder_block(d3, 64,  x1)

    output_layer = Conv2D(1, (1, 1), padding="same", activation="sigmoid")(d4)

    model = Model(input_layer, output_layer)
    return model


# ============================
# COMPILAR Y ENTRENAR
# ============================
model = build_model()
model.summary()

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

checkpoint_path = f"/kaggle/working/best_model_augprofile_{AUG_PROFILE}.keras"

checkpoint = ModelCheckpoint(
    checkpoint_path,
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    mode='min',
    verbose=1,
    restore_best_weights=True
)

EPOCHS = 100  # ajusta si necesitas menos

history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=test_dataset,
    callbacks=[checkpoint, early_stopping],
    verbose=1
)


# ============================
# GUARDAR HISTÓRICO COMPLETO DE ÉPOCAS EN CSV
# ============================
history_dict = history.history
num_epochs_run = len(history_dict["loss"])

rows = []
for epoch_idx in range(num_epochs_run):
    row = {
        "parameter": "aug_profile",
        "value": AUG_PROFILE,
        "epoch": epoch_idx + 1,
        "loss": history_dict["loss"][epoch_idx],
        "accuracy": history_dict["accuracy"][epoch_idx],
        "val_loss": history_dict["val_loss"][epoch_idx],
        "val_accuracy": history_dict["val_accuracy"][epoch_idx],
    }
    rows.append(row)

df_history = pd.DataFrame(rows)

history_csv_path = f"/kaggle/working/history_augprofile_{AUG_PROFILE}.csv"
df_history.to_csv(history_csv_path, index=False)

print("Histórico de épocas guardado en:", history_csv_path)
print(df_history.head())
print(df_history.tail())


## **Parámetro de OPTIMIZER CFG**

In [ ]:
# ============================
# CONFIGURACIÓN DEL EXPERIMENTO
# ============================

# Cambia SOLO esta línea entre ejecuciones:
#   "Adam"
#   "SGD_momentum"
#   "AdamW"

# OPTIMIZER_CFG = "Adam"
# OPTIMIZER_CFG = "SGD_momentum"
OPTIMIZER_CFG = "AdamW"

BASE_LR = 1e-3

print(f"Using OPTIMIZER_CFG = {OPTIMIZER_CFG}")


# ============================
# IMPORTS
# ============================
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.layers import (
    Dense, Conv2D, MaxPooling2D, Dropout, BatchNormalization,
    Conv2DTranspose, Activation, concatenate, Input
)
from tensorflow.keras import Model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

import matplotlib.pyplot as plt


# ============================
# RUTAS DEL DATASET EN KAGGLE
# ============================
BASE_PATH = "/kaggle/input/fundus-image-dataset-for-vessel-segmentation"
TRAIN_PATH = os.path.join(BASE_PATH, "train")
TEST_PATH  = os.path.join(BASE_PATH, "test")


# ============================
# CARGA DE DATOS
# ============================
def load_data(path, img_size):
    images = []
    masks = []
    num_images = 100  # ajustable

    img_dir = os.path.join(path, "Original")
    mask_dir = os.path.join(path, "Ground truth")

    img_files = sorted(os.listdir(img_dir))[:num_images]
    mask_files = sorted(os.listdir(mask_dir))[:num_images]

    for img_file, mask_file in zip(img_files, mask_files):
        img_path = os.path.join(img_dir, img_file)
        mask_path = os.path.join(mask_dir, mask_file)

        img = load_img(img_path, target_size=img_size)
        mask = load_img(mask_path, target_size=img_size, color_mode="grayscale")

        img = img_to_array(img) / 255.0
        mask = img_to_array(mask) / 255.0

        images.append(img)
        masks.append(mask)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)


IMG_SIZE = (512, 512)
x_train, y_train = load_data(TRAIN_PATH, IMG_SIZE)
x_test,  y_test  = load_data(TEST_PATH,  IMG_SIZE)

print("Train shapes:", x_train.shape, y_train.shape)
print("Test  shapes:", x_test.shape,  y_test.shape)


# ============================
# DATASET TF + AUGMENTACIÓN
# ============================
def augment_image(image, mask):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    return image, mask


def tf_dataset(x, y, batch_size=8, augment=True):
    dataset = tf.data.Dataset.from_tensor_slices((x, y))
    dataset = dataset.shuffle(buffer_size=len(x))

    if augment:
        dataset = dataset.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)

    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


BATCH_SIZE = 8
train_dataset = tf_dataset(x_train, y_train, batch_size=BATCH_SIZE, augment=True)
test_dataset  = tf_dataset(x_test,  y_test,  batch_size=BATCH_SIZE, augment=False)


# ============================
# DEFINICIÓN DEL MODELO (UNET BASE)
# ============================
def conv_block(input_tensor, filters):
    x = Conv2D(filters, 3, padding="same")(input_tensor)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = Conv2D(filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    return x


def encoder_block(input_tensor, filters):
    x = conv_block(input_tensor, filters)
    p = MaxPooling2D((2, 2))(x)
    return x, p


def decoder_block(input_tensor, filters, skip_tensor):
    x = Conv2DTranspose(filters, 2, strides=2, padding="same")(input_tensor)
    x = concatenate([x, skip_tensor])
    x = conv_block(x, filters)
    return x


def build_model():
    input_layer = Input(shape=(512, 512, 3))

    x1, p1 = encoder_block(input_layer, 64)
    x2, p2 = encoder_block(p1, 128)
    x3, p3 = encoder_block(p2, 256)
    x4, p4 = encoder_block(p3, 512)

    a1 = conv_block(p4, 1024)

    d1 = decoder_block(a1, 512, x4)
    d2 = decoder_block(d1, 256, x3)
    d3 = decoder_block(d2, 128, x2)
    d4 = decoder_block(d3, 64,  x1)

    output_layer = Conv2D(1, (1, 1), padding="same", activation="sigmoid")(d4)

    model = Model(input_layer, output_layer)
    return model


# ============================
# OPTIMIZER SEGÚN optimizer_cfg
# ============================
def get_optimizer(opt_name, base_lr):
    if opt_name == "Adam":
        return tf.keras.optimizers.Adam(learning_rate=base_lr)

    elif opt_name == "SGD_momentum":
        return tf.keras.optimizers.SGD(
            learning_rate=base_lr,
            momentum=0.9,
            nesterov=True
        )

    elif opt_name == "AdamW":
        # AdamW con un weight decay moderado
        return tf.keras.optimizers.AdamW(
            learning_rate=base_lr,
            weight_decay=1e-4
        )
    else:
        raise ValueError(f"Unknown OPTIMIZER_CFG: {opt_name}")


# ============================
# COMPILAR Y ENTRENAR
# ============================
model = build_model()
model.summary()

optimizer = get_optimizer(OPTIMIZER_CFG, BASE_LR)

model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"])

checkpoint_path = f"/kaggle/working/best_model_optcfg_{OPTIMIZER_CFG}.keras"

checkpoint = ModelCheckpoint(
    checkpoint_path,
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    mode='min',
    verbose=1,
    restore_best_weights=True
)

EPOCHS = 100  # ajusta si necesitas menos

history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=test_dataset,
    callbacks=[checkpoint, early_stopping],
    verbose=1
)


# ============================
# GUARDAR HISTÓRICO COMPLETO DE ÉPOCAS EN CSV
# ============================
history_dict = history.history
num_epochs_run = len(history_dict["loss"])

rows = []
for epoch_idx in range(num_epochs_run):
    row = {
        "parameter": "optimizer_cfg",
        "value": OPTIMIZER_CFG,
        "epoch": epoch_idx + 1,
        "loss": history_dict["loss"][epoch_idx],
        "accuracy": history_dict["accuracy"][epoch_idx],
        "val_loss": history_dict["val_loss"][epoch_idx],
        "val_accuracy": history_dict["val_accuracy"][epoch_idx],
    }
    rows.append(row)

df_history = pd.DataFrame(rows)

history_csv_path = f"/kaggle/working/history_optcfg_{OPTIMIZER_CFG}.csv"
df_history.to_csv(history_csv_path, index=False)

print("Histórico de épocas guardado en:", history_csv_path)
print(df_history.head())
print(df_history.tail())


## **Parámetro de KERNEL**

In [ ]:
# ============================
# CONFIGURACIÓN DEL EXPERIMENTO
# ============================

# Cambia SOLO esta línea entre ejecuciones:
#   3
#   5
#   7

# KERNEL_SIZE_CFG = 3
KERNEL_SIZE_CFG = 5
# KERNEL_SIZE_CFG = 7

print(f"Using KERNEL_SIZE_CFG = {KERNEL_SIZE_CFG}")


# ============================
# IMPORTS
# ============================
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.layers import (
    Dense, Conv2D, MaxPooling2D, Dropout, BatchNormalization,
    Conv2DTranspose, Activation, concatenate, Input
)
from tensorflow.keras import Model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

import matplotlib.pyplot as plt


# ============================
# RUTAS DEL DATASET EN KAGGLE
# ============================
# Dataset: nikitamanaenkov/fundus-image-dataset-for-vessel-segmentation
BASE_PATH = "/kaggle/input/fundus-image-dataset-for-vessel-segmentation"
TRAIN_PATH = os.path.join(BASE_PATH, "train")
TEST_PATH  = os.path.join(BASE_PATH, "test")


# ============================
# CARGA DE DATOS
# ============================
def load_data(path, img_size):
    images = []
    masks = []
    num_images = 100  # puedes ajustarlo para ir más rápido si quieres

    img_dir = os.path.join(path, "Original")
    mask_dir = os.path.join(path, "Ground truth")

    img_files = sorted(os.listdir(img_dir))[:num_images]
    mask_files = sorted(os.listdir(mask_dir))[:num_images]

    for img_file, mask_file in zip(img_files, mask_files):
        img_path = os.path.join(img_dir, img_file)
        mask_path = os.path.join(mask_dir, mask_file)

        img = load_img(img_path, target_size=img_size)
        mask = load_img(mask_path, target_size=img_size, color_mode="grayscale")

        img = img_to_array(img) / 255.0
        mask = img_to_array(mask) / 255.0

        images.append(img)
        masks.append(mask)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)


IMG_SIZE = (512, 512)
x_train, y_train = load_data(TRAIN_PATH, IMG_SIZE)
x_test,  y_test  = load_data(TEST_PATH,  IMG_SIZE)

print("Train shapes:", x_train.shape, y_train.shape)
print("Test  shapes:", x_test.shape,  y_test.shape)


# ============================
# DATASET TF + AUGMENTACIÓN
# ============================
def augment_image(image, mask):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    return image, mask


def tf_dataset(x, y, batch_size=8, augment=True):
    dataset = tf.data.Dataset.from_tensor_slices((x, y))
    dataset = dataset.shuffle(buffer_size=len(x))

    if augment:
        dataset = dataset.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)

    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


BATCH_SIZE = 8
train_dataset = tf_dataset(x_train, y_train, batch_size=BATCH_SIZE, augment=True)
test_dataset  = tf_dataset(x_test,  y_test,  batch_size=BATCH_SIZE, augment=False)


# ============================
# DEFINICIÓN DEL MODELO (UNET CON KERNEL VARIABLE)
# ============================
def conv_block(input_tensor, filters):
    k = KERNEL_SIZE_CFG  # usamos el parámetro global

    x = Conv2D(filters, k, padding="same")(input_tensor)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = Conv2D(filters, k, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    return x


def encoder_block(input_tensor, filters):
    x = conv_block(input_tensor, filters)
    p = MaxPooling2D((2, 2))(x)
    return x, p


def decoder_block(input_tensor, filters, skip_tensor):
    x = Conv2DTranspose(filters, 2, strides=2, padding="same")(input_tensor)
    x = concatenate([x, skip_tensor])
    x = conv_block(x, filters)
    return x


def build_model():
    input_layer = Input(shape=(512, 512, 3))

    x1, p1 = encoder_block(input_layer, 64)
    x2, p2 = encoder_block(p1, 128)
    x3, p3 = encoder_block(p2, 256)
    x4, p4 = encoder_block(p3, 512)

    a1 = conv_block(p4, 1024)

    d1 = decoder_block(a1, 512, x4)
    d2 = decoder_block(d1, 256, x3)
    d3 = decoder_block(d2, 128, x2)
    d4 = decoder_block(d3, 64,  x1)

    output_layer = Conv2D(1, (1, 1), padding="same", activation="sigmoid")(d4)

    model = Model(input_layer, output_layer)
    return model


# ============================
# COMPILAR Y ENTRENAR
# ============================
model = build_model()
model.summary()

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

checkpoint_path = f"/kaggle/working/best_model_kernel_{KERNEL_SIZE_CFG}.keras"

checkpoint = ModelCheckpoint(
    checkpoint_path,
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    mode='min',
    verbose=1,
    restore_best_weights=True
)

EPOCHS = 100  # puedes bajarlo si necesitas ir más rápido

history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=test_dataset,
    callbacks=[checkpoint, early_stopping],
    verbose=1
)


# ============================
# GUARDAR HISTÓRICO COMPLETO DE ÉPOCAS EN CSV
# ============================
history_dict = history.history
num_epochs_run = len(history_dict["loss"])

rows = []
for epoch_idx in range(num_epochs_run):
    row = {
        "parameter": "kernel_size",
        "value": KERNEL_SIZE_CFG,
        "epoch": epoch_idx + 1,
        "loss": history_dict["loss"][epoch_idx],
        "accuracy": history_dict["accuracy"][epoch_idx],
        "val_loss": history_dict["val_loss"][epoch_idx],
        "val_accuracy": history_dict["val_accuracy"][epoch_idx],
    }
    rows.append(row)

df_history = pd.DataFrame(rows)

history_csv_path = f"/kaggle/working/history_kernel_{KERNEL_SIZE_CFG}.csv"
df_history.to_csv(history_csv_path, index=False)

print("Histórico de épocas guardado en:", history_csv_path)
print(df_history.head())
print(df_history.tail())


## **Parámetro de NORM_LAYER**

In [ ]:
# ============================
# CONFIGURACIÓN DEL EXPERIMENTO
# ============================

# Cambia SOLO esta línea entre ejecuciones:
#   "BatchNorm"
#   "GroupNorm"
#   "None"

# NORM_LAYER_CFG = "BatchNorm"
# NORM_LAYER_CFG = "GroupNorm"
NORM_LAYER_CFG = "None"

print(f"Using NORM_LAYER_CFG = {NORM_LAYER_CFG}")


# ============================
# IMPORTS
# ============================
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, BatchNormalization,
    Conv2DTranspose, Activation, concatenate, Input
)
from tensorflow.keras import Model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

import matplotlib.pyplot as plt


# ============================
# RUTA DEL DATASET EN KAGGLE
# ============================
# Dataset: nikitamanaenkov/fundus-image-dataset-for-vessel-segmentation
BASE_PATH = "/kaggle/input/fundus-image-dataset-for-vessel-segmentation"
TRAIN_PATH = os.path.join(BASE_PATH, "train")
TEST_PATH  = os.path.join(BASE_PATH, "test")


# ============================
# CARGA DE DATOS
# ============================
def load_data(path, img_size):
    images = []
    masks = []
    num_images = 100  # ajusta si quieres entrenar más rápido

    img_dir = os.path.join(path, "Original")
    mask_dir = os.path.join(path, "Ground truth")

    img_files = sorted(os.listdir(img_dir))[:num_images]
    mask_files = sorted(os.listdir(mask_dir))[:num_images]

    for img_file, mask_file in zip(img_files, mask_files):
        img_path = os.path.join(img_dir, img_file)
        mask_path = os.path.join(mask_dir, mask_file)

        img = load_img(img_path, target_size=img_size)
        mask = load_img(mask_path, target_size=img_size, color_mode="grayscale")

        img = img_to_array(img) / 255.0
        mask = img_to_array(mask) / 255.0

        images.append(img)
        masks.append(mask)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)


IMG_SIZE = (512, 512)
x_train, y_train = load_data(TRAIN_PATH, IMG_SIZE)
x_test,  y_test  = load_data(TEST_PATH,  IMG_SIZE)

print("Train shapes:", x_train.shape, y_train.shape)
print("Test  shapes:", x_test.shape,  y_test.shape)


# ============================
# DATASET TF + AUGMENTACIÓN
# ============================
def augment_image(image, mask):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    return image, mask


def tf_dataset(x, y, batch_size=8, augment=True):
    dataset = tf.data.Dataset.from_tensor_slices((x, y))
    dataset = dataset.shuffle(buffer_size=len(x))

    if augment:
        dataset = dataset.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)

    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


BATCH_SIZE = 8
train_dataset = tf_dataset(x_train, y_train, batch_size=BATCH_SIZE, augment=True)
test_dataset  = tf_dataset(x_test,  y_test,  batch_size=BATCH_SIZE, augment=False)


# ============================
# CAPA GroupNorm (G = 8)
# ============================
class GroupNorm(tf.keras.layers.Layer):
    def __init__(self, groups=8, epsilon=1e-5, **kwargs):
        super().__init__(**kwargs)
        self.groups = groups
        self.epsilon = epsilon

    def build(self, input_shape):
        channels = int(input_shape[-1])
        self.groups = min(self.groups, channels)  # por seguridad
        self.gamma = self.add_weight(
            name="gamma",
            shape=(channels,),
            initializer="ones",
            trainable=True,
        )
        self.beta = self.add_weight(
            name="beta",
            shape=(channels,),
            initializer="zeros",
            trainable=True,
        )
        super().build(input_shape)

    def call(self, x):
        # x: [N, H, W, C]
        N, H, W, C = tf.unstack(tf.shape(x))
        G = self.groups
        x = tf.reshape(x, [N, H, W, G, C // G])

        mean, var = tf.nn.moments(x, axes=[1, 2, 4], keepdims=True)
        x = (x - mean) / tf.sqrt(var + self.epsilon)

        x = tf.reshape(x, [N, H, W, C])
        return self.gamma * x + self.beta


def apply_norm(x):
    if NORM_LAYER_CFG == "BatchNorm":
        return BatchNormalization()(x)
    elif NORM_LAYER_CFG == "GroupNorm":
        return GroupNorm(groups=8)(x)
    elif NORM_LAYER_CFG == "None":
        return x
    else:
        raise ValueError(f"Unknown NORM_LAYER_CFG: {NORM_LAYER_CFG}")


# ============================
# DEFINICIÓN DEL MODELO (UNET)
# ============================
def conv_block(input_tensor, filters):
    x = Conv2D(filters, 3, padding="same")(input_tensor)
    x = apply_norm(x)
    x = Activation("relu")(x)

    x = Conv2D(filters, 3, padding="same")(x)
    x = apply_norm(x)
    x = Activation("relu")(x)
    return x


def encoder_block(input_tensor, filters):
    x = conv_block(input_tensor, filters)
    p = MaxPooling2D((2, 2))(x)
    return x, p


def decoder_block(input_tensor, filters, skip_tensor):
    x = Conv2DTranspose(filters, 2, strides=2, padding="same")(input_tensor)
    x = concatenate([x, skip_tensor])
    x = conv_block(x, filters)
    return x


def build_model():
    input_layer = Input(shape=(512, 512, 3))

    x1, p1 = encoder_block(input_layer, 64)
    x2, p2 = encoder_block(p1, 128)
    x3, p3 = encoder_block(p2, 256)
    x4, p4 = encoder_block(p3, 512)

    a1 = conv_block(p4, 1024)

    d1 = decoder_block(a1, 512, x4)
    d2 = decoder_block(d1, 256, x3)
    d3 = decoder_block(d2, 128, x2)
    d4 = decoder_block(d3, 64,  x1)

    output_layer = Conv2D(1, (1, 1), padding="same", activation="sigmoid")(d4)

    model = Model(input_layer, output_layer)
    return model


# ============================
# COMPILAR Y ENTRENAR
# ============================
model = build_model()
model.summary()

model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=["accuracy"])

checkpoint_path = f"/kaggle/working/best_model_normlayer_{NORM_LAYER_CFG}.keras"

checkpoint = ModelCheckpoint(
    checkpoint_path,
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    mode='min',
    verbose=1,
    restore_best_weights=True
)

EPOCHS = 100  # ajusta si quieres menos

history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=test_dataset,
    callbacks=[checkpoint, early_stopping],
    verbose=1
)


# ============================
# GUARDAR HISTÓRICO COMPLETO EN CSV
# ============================
history_dict = history.history
num_epochs_run = len(history_dict["loss"])

rows = []
for epoch_idx in range(num_epochs_run):
    row = {
        "parameter": "norm_layer",
        "value": NORM_LAYER_CFG,
        "epoch": epoch_idx + 1,
        "loss": history_dict["loss"][epoch_idx],
        "accuracy": history_dict["accuracy"][epoch_idx],
        "val_loss": history_dict["val_loss"][epoch_idx],
        "val_accuracy": history_dict["val_accuracy"][epoch_idx],
    }
    rows.append(row)

df_history = pd.DataFrame(rows)

history_csv_path = f"/kaggle/working/history_normlayer_{NORM_LAYER_CFG}.csv"
df_history.to_csv(history_csv_path, index=False)

print("Histórico de épocas guardado en:", history_csv_path)
print(df_history.head())
print(df_history.tail())


## **Parámetro de DROP_L2**

In [ ]:
# ============================
# CONFIGURACIÓN DEL EXPERIMENTO
# ============================

# Cambia SOLO esta línea entre ejecuciones:
#   "no_reg"     -> dropout 0.0, L2 0
#   "mild_reg"   -> dropout 0.2, L2 1e-5
#   "strong_reg" -> dropout 0.4, L2 1e-4

# DROP_L2_CFG = "no_reg"
# DROP_L2_CFG = "mild_reg"
DROP_L2_CFG = "strong_reg"

print(f"Using DROP_L2_CFG = {DROP_L2_CFG}")


# ============================
# IMPORTS
# ============================
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, BatchNormalization,
    Conv2DTranspose, Activation, concatenate, Input,
    Dropout
)
from tensorflow.keras import Model, regularizers
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

import matplotlib.pyplot as plt


# ============================
# MAPEO CONFIG -> (dropout, L2)
# ============================
def get_dropout_l2(cfg_name):
    if cfg_name == "no_reg":
        return 0.0, 0.0
    elif cfg_name == "mild_reg":
        return 0.2, 1e-5
    elif cfg_name == "strong_reg":
        return 0.4, 1e-4
    else:
        raise ValueError(f"Unknown DROP_L2_CFG: {cfg_name}")


DROPOUT_RATE, L2_WEIGHT = get_dropout_l2(DROP_L2_CFG)
print(f"Dropout rate = {DROPOUT_RATE}, L2 = {L2_WEIGHT}")


# ============================
# RUTA DEL DATASET EN KAGGLE
# ============================
BASE_PATH = "/kaggle/input/fundus-image-dataset-for-vessel-segmentation"
TRAIN_PATH = os.path.join(BASE_PATH, "train")
TEST_PATH  = os.path.join(BASE_PATH, "test")


# ============================
# CARGA DE DATOS
# ============================
def load_data(path, img_size):
    images = []
    masks = []
    num_images = 100  # ajusta si quieres entrenar más rápido

    img_dir = os.path.join(path, "Original")
    mask_dir = os.path.join(path, "Ground truth")

    img_files = sorted(os.listdir(img_dir))[:num_images]
    mask_files = sorted(os.listdir(mask_dir))[:num_images]

    for img_file, mask_file in zip(img_files, mask_files):
        img_path = os.path.join(img_dir, img_file)
        mask_path = os.path.join(mask_dir, mask_file)

        img = load_img(img_path, target_size=img_size)
        mask = load_img(mask_path, target_size=img_size, color_mode="grayscale")

        img = img_to_array(img) / 255.0
        mask = img_to_array(mask) / 255.0

        images.append(img)
        masks.append(mask)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)


IMG_SIZE = (512, 512)
x_train, y_train = load_data(TRAIN_PATH, IMG_SIZE)
x_test,  y_test  = load_data(TEST_PATH,  IMG_SIZE)

print("Train shapes:", x_train.shape, y_train.shape)
print("Test  shapes:", x_test.shape,  y_test.shape)


# ============================
# DATASET TF + AUGMENTACIÓN
# ============================
def augment_image(image, mask):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    return image, mask


def tf_dataset(x, y, batch_size=8, augment=True):
    dataset = tf.data.Dataset.from_tensor_slices((x, y))
    dataset = dataset.shuffle(buffer_size=len(x))

    if augment:
        dataset = dataset.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)

    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


BATCH_SIZE = 8
train_dataset = tf_dataset(x_train, y_train, batch_size=BATCH_SIZE, augment=True)
test_dataset  = tf_dataset(x_test,  y_test,  batch_size=BATCH_SIZE, augment=False)


# ============================
# DEFINICIÓN DEL MODELO (UNET con Dropout + L2)
# ============================
def conv_block(input_tensor, filters):
    # L2 regularizer (o None si peso = 0)
    kernel_reg = regularizers.l2(L2_WEIGHT) if L2_WEIGHT > 0.0 else None

    x = Conv2D(filters, 3, padding="same",
               kernel_regularizer=kernel_reg)(input_tensor)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    if DROPOUT_RATE > 0.0:
        x = Dropout(DROPOUT_RATE)(x)

    x = Conv2D(filters, 3, padding="same",
               kernel_regularizer=kernel_reg)(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    if DROPOUT_RATE > 0.0:
        x = Dropout(DROPOUT_RATE)(x)

    return x


def encoder_block(input_tensor, filters):
    x = conv_block(input_tensor, filters)
    p = MaxPooling2D((2, 2))(x)
    return x, p


def decoder_block(input_tensor, filters, skip_tensor):
    x = Conv2DTranspose(filters, 2, strides=2, padding="same")(input_tensor)
    x = concatenate([x, skip_tensor])
    x = conv_block(x, filters)
    return x


def build_model():
    input_layer = Input(shape=(512, 512, 3))

    x1, p1 = encoder_block(input_layer, 64)
    x2, p2 = encoder_block(p1, 128)
    x3, p3 = encoder_block(p2, 256)
    x4, p4 = encoder_block(p3, 512)

    a1 = conv_block(p4, 1024)

    d1 = decoder_block(a1, 512, x4)
    d2 = decoder_block(d1, 256, x3)
    d3 = decoder_block(d2, 128, x2)
    d4 = decoder_block(d3, 64,  x1)

    output_layer = Conv2D(1, (1, 1), padding="same", activation="sigmoid")(d4)

    model = Model(input_layer, output_layer)
    return model


# ============================
# COMPILAR Y ENTRENAR
# ============================
model = build_model()
model.summary()

model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=["accuracy"])

checkpoint_path = f"/kaggle/working/best_model_dropl2_{DROP_L2_CFG}.keras"

checkpoint = ModelCheckpoint(
    checkpoint_path,
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    mode='min',
    verbose=1,
    restore_best_weights=True
)

EPOCHS = 100  # reduce si necesitas ir más rápido

history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=test_dataset,
    callbacks=[checkpoint, early_stopping],
    verbose=1
)


# ============================
# GUARDAR HISTÓRICO COMPLETO EN CSV
# ============================
history_dict = history.history
num_epochs_run = len(history_dict["loss"])

rows = []
for epoch_idx in range(num_epochs_run):
    row = {
        "parameter": "dropout_L2",
        "value": DROP_L2_CFG,
        "epoch": epoch_idx + 1,
        "loss": history_dict["loss"][epoch_idx],
        "accuracy": history_dict["accuracy"][epoch_idx],
        "val_loss": history_dict["val_loss"][epoch_idx],
        "val_accuracy": history_dict["val_accuracy"][epoch_idx],
    }
    rows.append(row)

df_history = pd.DataFrame(rows)

history_csv_path = f"/kaggle/working/history_dropl2_{DROP_L2_CFG}.csv"
df_history.to_csv(history_csv_path, index=False)

print("Histórico de épocas guardado en:", history_csv_path)
print(df_history.head())
print(df_history.tail())


## **Parámetro de DOWN_OP**

In [ ]:
# ============================
# CONFIGURACIÓN DEL EXPERIMENTO
# ============================

# Cambia SOLO esta línea entre ejecuciones:
#   "MaxPool"
#   "AvgPool"
#   "StridedConv"

# DOWN_OP_CFG = "MaxPool"
# DOWN_OP_CFG = "AvgPool"
DOWN_OP_CFG = "StridedConv"

print(f"Using DOWN_OP_CFG = {DOWN_OP_CFG}")


# ============================
# IMPORTS
# ============================
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, AveragePooling2D, BatchNormalization,
    Conv2DTranspose, Activation, concatenate, Input
)
from tensorflow.keras import Model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

import matplotlib.pyplot as plt


# ============================
# RUTA DEL DATASET EN KAGGLE
# ============================
BASE_PATH = "/kaggle/input/fundus-image-dataset-for-vessel-segmentation"
TRAIN_PATH = os.path.join(BASE_PATH, "train")
TEST_PATH  = os.path.join(BASE_PATH, "test")


# ============================
# CARGA DE DATOS
# ============================
def load_data(path, img_size):
    images = []
    masks = []
    num_images = 100  # ajusta si quieres entrenar más rápido

    img_dir = os.path.join(path, "Original")
    mask_dir = os.path.join(path, "Ground truth")

    img_files = sorted(os.listdir(img_dir))[:num_images]
    mask_files = sorted(os.listdir(mask_dir))[:num_images]

    for img_file, mask_file in zip(img_files, mask_files):
        img_path = os.path.join(img_dir, img_file)
        mask_path = os.path.join(mask_dir, mask_file)

        img = load_img(img_path, target_size=img_size)
        mask = load_img(mask_path, target_size=img_size, color_mode="grayscale")

        img = img_to_array(img) / 255.0
        mask = img_to_array(mask) / 255.0

        images.append(img)
        masks.append(mask)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)


IMG_SIZE = (512, 512)
x_train, y_train = load_data(TRAIN_PATH, IMG_SIZE)
x_test,  y_test  = load_data(TEST_PATH,  IMG_SIZE)

print("Train shapes:", x_train.shape, y_train.shape)
print("Test  shapes:", x_test.shape,  y_test.shape)


# ============================
# DATASET TF + AUGMENTACIÓN
# ============================
def augment_image(image, mask):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    return image, mask


def tf_dataset(x, y, batch_size=8, augment=True):
    dataset = tf.data.Dataset.from_tensor_slices((x, y))
    dataset = dataset.shuffle(buffer_size=len(x))

    if augment:
        dataset = dataset.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)

    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


BATCH_SIZE = 8
train_dataset = tf_dataset(x_train, y_train, batch_size=BATCH_SIZE, augment=True)
test_dataset  = tf_dataset(x_test,  y_test,  batch_size=BATCH_SIZE, augment=False)


# ============================
# BLOQUE DE CONVOLUCIÓN BÁSICO
# ============================
def conv_block(input_tensor, filters):
    x = Conv2D(filters, 3, padding="same")(input_tensor)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = Conv2D(filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    return x


# ============================
# OPERACIÓN DE DOWN-SAMPLING
# ============================
def downsample_op(x, filters):
    """
    Aplica la operación de downsampling según DOWN_OP_CFG.
    Devuelve solo el tensor downsampleado (la skip va aparte).
    """
    if DOWN_OP_CFG == "MaxPool":
        p = MaxPooling2D((2, 2))(x)
    elif DOWN_OP_CFG == "AvgPool":
        p = AveragePooling2D((2, 2))(x)
    elif DOWN_OP_CFG == "StridedConv":
        # Conv 3x3 con stride=2 para reducir resolución
        p = Conv2D(filters, 3, strides=2, padding="same")(x)
        p = BatchNormalization()(p)
        p = Activation("relu")(p)
    else:
        raise ValueError(f"Unknown DOWN_OP_CFG: {DOWN_OP_CFG}")
    return p


def encoder_block(input_tensor, filters):
    x = conv_block(input_tensor, filters)
    p = downsample_op(x, filters)
    return x, p


# ============================
# BLOQUE DE DECODIFICACIÓN (UP-SAMPLING)
# ============================
def decoder_block(input_tensor, filters, skip_tensor):
    x = Conv2DTranspose(filters, 2, strides=2, padding="same")(input_tensor)
    x = concatenate([x, skip_tensor])
    x = conv_block(x, filters)
    return x


# ============================
# DEFINICIÓN DEL MODELO UNET
# ============================
def build_model():
    input_layer = Input(shape=(512, 512, 3))

    x1, p1 = encoder_block(input_layer, 64)
    x2, p2 = encoder_block(p1, 128)
    x3, p3 = encoder_block(p2, 256)
    x4, p4 = encoder_block(p3, 512)

    a1 = conv_block(p4, 1024)

    d1 = decoder_block(a1, 512, x4)
    d2 = decoder_block(d1, 256, x3)
    d3 = decoder_block(d2, 128, x2)
    d4 = decoder_block(d3, 64,  x1)

    output_layer = Conv2D(1, (1, 1), padding="same", activation="sigmoid")(d4)

    model = Model(input_layer, output_layer)
    return model


# ============================
# COMPILAR Y ENTRENAR
# ============================
model = build_model()
model.summary()

model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=["accuracy"])

checkpoint_path = f"/kaggle/working/best_model_downop_{DOWN_OP_CFG}.keras"

checkpoint = ModelCheckpoint(
    checkpoint_path,
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    mode='min',
    verbose=1,
    restore_best_weights=True
)

EPOCHS = 100  # ajusta si quieres menos

history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=test_dataset,
    callbacks=[checkpoint, early_stopping],
    verbose=1
)


# ============================
# GUARDAR HISTÓRICO COMPLETO EN CSV
# ============================
history_dict = history.history
num_epochs_run = len(history_dict["loss"])

rows = []
for epoch_idx in range(num_epochs_run):
    row = {
        "parameter": "down_op",
        "value": DOWN_OP_CFG,
        "epoch": epoch_idx + 1,
        "loss": history_dict["loss"][epoch_idx],
        "accuracy": history_dict["accuracy"][epoch_idx],
        "val_loss": history_dict["val_loss"][epoch_idx],
        "val_accuracy": history_dict["val_accuracy"][epoch_idx],
    }
    rows.append(row)

df_history = pd.DataFrame(rows)

history_csv_path = f"/kaggle/working/history_downop_{DOWN_OP_CFG}.csv"
df_history.to_csv(history_csv_path, index=False)

print("Histórico de épocas guardado en:", history_csv_path)
print(df_history.head())
print(df_history.tail())


## **Parámetro de WIDTH**

In [ ]:
# ============================
# CONFIGURACIÓN DEL EXPERIMENTO
# ============================

# Cambia SOLO esta línea entre ejecuciones:
WIDTH_CFG = 32
# WIDTH_CFG = 64
# WIDTH_CFG = 96


# ============================
# IMPORTS
# ============================
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Dropout, BatchNormalization,
    Conv2DTranspose, Activation, concatenate, Input
)
from tensorflow.keras import Model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

import matplotlib.pyplot as plt


# ============================
# RUTAS DEL DATASET EN KAGGLE
# ============================
BASE_PATH = "/kaggle/input/fundus-image-dataset-for-vessel-segmentation"
TRAIN_PATH = os.path.join(BASE_PATH, "train")
TEST_PATH  = os.path.join(BASE_PATH, "test")


# ============================
# CARGA DE DATOS
# ============================
def load_data(path, img_size):
    images = []
    masks = []
    num_images = 100

    img_dir = os.path.join(path, "Original")
    mask_dir = os.path.join(path, "Ground truth")

    img_files = sorted(os.listdir(img_dir))[:num_images]
    mask_files = sorted(os.listdir(mask_dir))[:num_images]

    for img_file, mask_file in zip(img_files, mask_files):
        img_path = os.path.join(img_dir, img_file)
        mask_path = os.path.join(mask_dir, mask_file)

        img = load_img(img_path, target_size=img_size)
        mask = load_img(mask_path, target_size=img_size, color_mode="grayscale")

        img = img_to_array(img) / 255.0
        mask = img_to_array(mask) / 255.0

        images.append(img)
        masks.append(mask)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)


IMG_SIZE = (512, 512)
x_train, y_train = load_data(TRAIN_PATH, IMG_SIZE)
x_test,  y_test  = load_data(TEST_PATH,  IMG_SIZE)


# ============================
# DATASET + AUGMENTACIÓN
# ============================
def augment_image(image, mask):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    return image, mask


def tf_dataset(x, y, batch_size=8, augment=True):
    dataset = tf.data.Dataset.from_tensor_slices((x, y))
    dataset = dataset.shuffle(len(x))

    if augment:
        dataset = dataset.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)

    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


train_dataset = tf_dataset(x_train, y_train, batch_size=8, augment=True)
test_dataset  = tf_dataset(x_test,  y_test,  batch_size=8, augment=False)


# ============================
# FUNCIÓN DE PÉRDIDA (misma que el profe)
# ============================
def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    numerator = 2.0 * intersection + smooth
    denominator = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth
    dice = numerator / denominator
    return 1.0 - dice

bce_loss_fn = tf.keras.losses.BinaryCrossentropy()

def bce_plus_dice_loss(y_true, y_pred):
    return bce_loss_fn(y_true, y_pred) + dice_loss(y_true, y_pred)

# En este experimento usamos siempre BCE + Dice
loss_fn = bce_plus_dice_loss


# ============================
# MODELO UNET CON WIDTH_CFG
# ============================
def conv_block(x, filters):
    x = Conv2D(filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = Conv2D(filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    return x


def encoder_block(x, filters):
    c = conv_block(x, filters)
    p = MaxPooling2D((2, 2))(c)
    return c, p


def decoder_block(x, filters, skip):
    x = Conv2DTranspose(filters, 2, strides=2, padding="same")(x)
    x = concatenate([x, skip])
    x = conv_block(x, filters)
    return x


def build_model(width):
    inp = Input(shape=(512, 512, 3))

    f1, f2, f3, f4, f5 = width, width*2, width*4, width*8, width*16

    x1, p1 = encoder_block(inp, f1)
    x2, p2 = encoder_block(p1, f2)
    x3, p3 = encoder_block(p2, f3)
    x4, p4 = encoder_block(p3, f4)

    bn = conv_block(p4, f5)

    d1 = decoder_block(bn, f4, x4)
    d2 = decoder_block(d1, f3, x3)
    d3 = decoder_block(d2, f2, x2)
    d4 = decoder_block(d3, f1, x1)

    out = Conv2D(1, 1, activation="sigmoid")(d4)

    return Model(inp, out)


# ============================
# COMPILAR Y ENTRENAR
# ============================
model = build_model(WIDTH_CFG)
model.summary()

model.compile(optimizer="adam", loss=loss_fn, metrics=["accuracy"])

checkpoint_path = f"/kaggle/working/best_model_width_{WIDTH_CFG}.keras"

checkpoint = ModelCheckpoint(
    checkpoint_path, monitor="val_loss",
    save_best_only=True, mode="min", verbose=1
)

early_stopping = EarlyStopping(
    monitor="val_loss", patience=15,
    mode="min", restore_best_weights=True, verbose=1
)

history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=100,
    callbacks=[checkpoint, early_stopping],
    verbose=1
)


# ============================
# GUARDAR HISTÓRICO EN CSV
# ============================
hist = history.history
epochs = len(hist["loss"])

rows = []
for e in range(epochs):
    rows.append({
        "parameter": "width_multiplier",
        "value": WIDTH_CFG,
        "epoch": e+1,
        "loss": hist["loss"][e],
        "accuracy": hist["accuracy"][e],
        "val_loss": hist["val_loss"][e],
        "val_accuracy": hist["val_accuracy"][e],
    })

df = pd.DataFrame(rows)
csv_path = f"/kaggle/working/history_width_{WIDTH_CFG}.csv"
df.to_csv(csv_path, index=False)

print("Histórico guardado en:", csv_path)
print(df.head())
print(df.tail())


## **Parámetro de DOWN_OP**

In [ ]:
# ============================
# CONFIGURACIÓN DEL EXPERIMENTO
# ============================

# Cambia SOLO esta línea entre ejecuciones:
DOWN_OP = "maxpool"
# DOWN_OP = "avgpool"
# DOWN_OP = "strided"

# ============================
# IMPORTS
# ============================
import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, AveragePooling2D, BatchNormalization,
    Conv2DTranspose, Activation, Add, Input, concatenate
)
from tensorflow.keras import Model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

import matplotlib.pyplot as plt



# ============================
# DATASET
# ============================
BASE_PATH = "/kaggle/input/fundus-image-dataset-for-vessel-segmentation"
TRAIN_PATH = os.path.join(BASE_PATH, "train")
TEST_PATH  = os.path.join(BASE_PATH, "test")

def load_data(path, img_size):
    images, masks = [], []
    num_images = 60  

    img_files = sorted(os.listdir(os.path.join(path, "Original")))[:num_images]
    mask_files = sorted(os.listdir(os.path.join(path, "Ground truth")))[:num_images]
 
    for img_file, mask_file in zip(img_files, mask_files):
        img = load_img(os.path.join(path, "Original", img_file), target_size=img_size)
        mask = load_img(os.path.join(path, "Ground truth", mask_file),
                        target_size=img_size, color_mode="grayscale")

        images.append(img_to_array(img)/255.0)
        masks.append(img_to_array(mask)/255.0)

    return np.array(images), np.array(masks)

IMG_SIZE = (512, 512)
x_train, y_train = load_data(TRAIN_PATH, IMG_SIZE)
x_test,  y_test  = load_data(TEST_PATH,  IMG_SIZE)



# ============================
# DATASET + AUGMENTACIÓN
# ============================
def augment_image(image, mask):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, 0.1)
    image = tf.image.random_contrast(image, 0.8, 1.2)
    return image, mask

def tf_dataset(x, y, batch=6, augment=True):
    ds = tf.data.Dataset.from_tensor_slices((x, y)).shuffle(len(x))
    if augment:
        ds = ds.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)

train_dataset = tf_dataset(x_train, y_train)
test_dataset  = tf_dataset(x_test,  y_test, augment=False)



# ============================
# MÉTRICAS
# ============================
def dice_metric(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    inter = tf.reduce_sum(y_true_f * y_pred_f)
    return (2*inter + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)



# ============================
# UNET con downsampling variable
# ============================
def downsample(x):
    if DOWN_OP == "maxpool":
        return MaxPooling2D(2)(x)
    elif DOWN_OP == "avgpool":
        return AveragePooling2D(2)(x)
    elif DOWN_OP == "strided":
        return Conv2D(x.shape[-1], 3, strides=2, padding="same")(x)
    else:
        raise ValueError("DOWN_OP inválido")


def conv_block(x, filters):
    x = Conv2D(filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = Conv2D(filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    return x


def encoder_block(x, filters):
    c = conv_block(x, filters)
    p = downsample(c)
    return c, p

def decoder_block(x, filters, skip):
    x = Conv2DTranspose(filters, 2, strides=2, padding="same")(x)
    x = concatenate([x, skip])
    x = conv_block(x, filters)
    return x


def build_model():
    inp = Input((512, 512, 3))

    x1, p1 = encoder_block(inp, 64)
    x2, p2 = encoder_block(p1, 128)
    x3, p3 = encoder_block(p2, 256)
    x4, p4 = encoder_block(p3, 512)

    b = conv_block(p4, 1024)

    d1 = decoder_block(b, 512, x4)
    d2 = decoder_block(d1, 256, x3)
    d3 = decoder_block(d2, 128, x2)
    d4 = decoder_block(d3, 64,  x1)

    out = Conv2D(1, 1, activation="sigmoid")(d4)

    return Model(inp, out)



# ============================
# ENTRENAMIENTO
# ============================
model = build_model()

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[dice_metric]
)

checkpoint = ModelCheckpoint(
    f"/kaggle/working/best_down_{DOWN_OP}.keras",
    monitor="val_loss",
    save_best_only=True
)

early = EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True
)

EPOCHS = 100
history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=EPOCHS,
    callbacks=[checkpoint, early],
    verbose=1
)



# ============================
# GRÁFICA COMBINADA LOSS + DICE
# ============================
plt.figure(figsize=(14,6))
epochs = range(1, len(history.history["loss"])+1)

fig, ax1 = plt.subplots(figsize=(14,6))

# EJE 1 → Pérdida
ax1.plot(epochs, history.history["loss"], label="loss", linewidth=2)
ax1.plot(epochs, history.history["val_loss"], label="val_loss", linewidth=2)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend(loc="upper left")

# EJE 2 → Métrica (dice)
ax2 = ax1.twinx()
ax2.plot(epochs, history.history["dice_metric"], "--", label="dice", linewidth=2)
ax2.plot(epochs, history.history["val_dice_metric"], "--", label="val_dice", linewidth=2)
ax2.set_ylabel("Dice")
ax2.legend(loc="upper right")

plt.title(f"Curvas Loss + Dice en SAME GRAPH – Downsampling={DOWN_OP}")
plt.show()



# ============================
# EJEMPLO DE CONTORNOS
# ============================
pred = model.predict(x_test[:1])[0,:,:,0]
plt.figure(figsize=(12,4))
plt.subplot(1,3,1)
plt.imshow(x_test[0])
plt.title("Imagen")

plt.subplot(1,3,2)
plt.imshow(y_test[0].squeeze(), cmap="gray")
plt.title("GT")

plt.subplot(1,3,3)
plt.imshow(pred, cmap="gray")
plt.title(f"Pred – {DOWN_OP}")
plt.show()


## **Parámetro de BLOCK TYPE**

In [ ]:
# ============================================
# CONFIGURACIÓN DEL EXPERIMENTO
# ============================================

# BLOCK_TYPE = "standard" 
# BLOCK_TYPE = "residual"
BLOCK_TYPE = "inverted"


# ============================================
# IMPORTS
# ============================================
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.layers import (
    Dense, Conv2D, MaxPooling2D, Dropout, BatchNormalization,
    Conv2DTranspose, Activation, concatenate, Input, Add
)
from tensorflow.keras import Model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

import matplotlib.pyplot as plt


# ============================================
# DATASET
# ============================================
BASE_PATH = "/kaggle/input/fundus-image-dataset-for-vessel-segmentation"

def load_data(path, img_size):
    img_dir = os.path.join(path, "Original")
    mask_dir = os.path.join(path, "Ground truth")

    images, masks = [], []
    files = sorted(os.listdir(img_dir))
    files = files[:100]

    for f in files:
        img = load_img(os.path.join(img_dir, f), target_size=img_size)
        mask = load_img(os.path.join(mask_dir, f), target_size=img_size, color_mode="grayscale")

        images.append(img_to_array(img)/255.0)
        masks.append(img_to_array(mask)/255.0)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)

IMG_SIZE = (512,512)
x_train, y_train = load_data(os.path.join(BASE_PATH, "train"), IMG_SIZE)
x_test,  y_test  = load_data(os.path.join(BASE_PATH, "test"),  IMG_SIZE)


def augment_image(image, mask):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, 0.1)
    return image, mask

def tf_dataset(x, y, batch=8, aug=True):
    ds = tf.data.Dataset.from_tensor_slices((x, y)).shuffle(len(x))
    if aug: ds = ds.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)

train_dataset = tf_dataset(x_train, y_train, batch=4, aug=True)
test_dataset  = tf_dataset(x_test,  y_test,  batch=4, aug=False)


# ============================================
# BLOQUES CONVOLUCIONALES SEGÚN BLOCK_TYPE
# ============================================

def block_standard(x, filters):
    y = Conv2D(filters, 3, padding="same")(x)
    y = BatchNormalization()(y)
    y = Activation("relu")(y)

    y = Conv2D(filters, 3, padding="same")(y)
    y = BatchNormalization()(y)
    y = Activation("relu")(y)
    return y


def block_residual(x, filters):
    shortcut = x
    if x.shape[-1] != filters:
        shortcut = Conv2D(filters, 1, padding="same")(x)

    y = Conv2D(filters, 3, padding="same")(x)
    y = BatchNormalization()(y)
    y = Activation("relu")(y)

    y = Conv2D(filters, 3, padding="same")(y)
    y = BatchNormalization()(y)

    y = Add()([shortcut, y])
    y = Activation("relu")(y)
    return y


def block_inverted(x, filters):
    expand = filters * 4

    y = Conv2D(expand, 1, padding="same")(x)
    y = BatchNormalization()(y)
    y = Activation("relu")(y)

    y = Conv2D(expand, 3, padding="same", groups=expand)(y)
    y = BatchNormalization()(y)
    y = Activation("relu")(y)

    y = Conv2D(filters, 1, padding="same")(y)
    y = BatchNormalization()(y)

    if x.shape[-1] == filters:
        y = Add()([x, y])
    return Activation("relu")(y)


# Selección dinámica del bloque
def get_block_fn():
    if BLOCK_TYPE == "standard":  return block_standard
    if BLOCK_TYPE == "residual":  return block_residual
    if BLOCK_TYPE == "inverted":  return block_inverted
    raise ValueError("BLOCK_TYPE inválido")


# ============================================
# UNET MODULAR
# ============================================

def encoder_block(x, filters, block_fn):
    x = block_fn(x, filters)
    p = MaxPooling2D(2)(x)
    return x, p

def decoder_block(x, filters, skip, block_fn):
    x = Conv2DTranspose(filters, 2, strides=2, padding="same")(x)
    x = concatenate([x, skip])
    x = block_fn(x, filters)
    return x


def build_model():
    block_fn = get_block_fn()

    inp = Input((512,512,3))

    x1, p1 = encoder_block(inp, 64, block_fn)
    x2, p2 = encoder_block(p1, 128, block_fn)
    x3, p3 = encoder_block(p2, 256, block_fn)
    x4, p4 = encoder_block(p3, 512, block_fn)

    b = block_fn(p4, 1024)

    d1 = decoder_block(b, 512, x4, block_fn)
    d2 = decoder_block(d1, 256, x3, block_fn)
    d3 = decoder_block(d2, 128, x2, block_fn)
    d4 = decoder_block(d3, 64,  x1, block_fn)

    out = Conv2D(1, 1, activation="sigmoid")(d4)

    return Model(inp, out)


# ============================================
# LOSS + TRAINING
# ============================================
bce = tf.keras.losses.BinaryCrossentropy()

def dice_loss(y_true, y_pred, eps=1e-6):
    t = tf.reshape(y_true, [-1])
    p = tf.reshape(y_pred, [-1])
    inter = tf.reduce_sum(t*p)
    return 1 - (2*inter + eps)/(tf.reduce_sum(t) + tf.reduce_sum(p) + eps)

def bce_plus_dice(y_true, y_pred):
    return bce(y_true,y_pred) + dice_loss(y_true,y_pred)

loss_fn = bce_plus_dice

model = build_model()
model.compile(optimizer="adam", loss=loss_fn, metrics=["accuracy"])

history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=100,
    verbose=1
)


# ============================================
# GUARDAR HISTÓRICO EN CSV
# ============================================
rows=[]
for e in range(len(history.history["loss"])):
    rows.append({
        "parameter": "block_type",
        "value": BLOCK_TYPE,
        "epoch": e+1,
        "loss": history.history["loss"][e],
        "accuracy": history.history["accuracy"][e],
        "val_loss": history.history["val_loss"][e],
        "val_accuracy": history.history["val_accuracy"][e]
    })

df = pd.DataFrame(rows)
csv_path = f"/kaggle/working/history_blocktype_{BLOCK_TYPE}.csv"
df.to_csv(csv_path, index=False)
print("Guardado en →", csv_path)


# ============================================
# GRAFICA LOSS + ACCURACY
# ============================================
plt.figure(figsize=(12,6))

plt.plot(df["epoch"], df["loss"], label="train_loss")
plt.plot(df["epoch"], df["val_loss"], label="val_loss")
plt.plot(df["epoch"], df["accuracy"], label="train_acc")
plt.plot(df["epoch"], df["val_accuracy"], label="val_acc")

plt.title(f"Curvas entrenamiento — BLOCK_TYPE={BLOCK_TYPE}")
plt.xlabel("Epoch")
plt.ylabel("Valor")
plt.legend()
plt.grid()
plt.show()


## **Parámetro de DEPTH**

In [ ]:
# ============================
# CONFIGURACIÓN DEL EXPERIMENTO 
# ============================

DEPTH_CFG = 3
# DEPTH_CFG = 4
# DEPTH_CFG = 5

# ============================
# IMPORTS
# ============================
import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.layers import (
    Dense, Conv2D, MaxPooling2D, Dropout, BatchNormalization,
    Conv2DTranspose, Activation, concatenate, Input
)
from tensorflow.keras import Model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, Callback

# ============================
# RUTAS DEL DATASET (ajusta si usas Colab local path)
# ============================
BASE_PATH = "/kaggle/input/fundus-image-dataset-for-vessel-segmentation"
TRAIN_PATH = os.path.join(BASE_PATH, "train")
TEST_PATH  = os.path.join(BASE_PATH, "test")

# ============================
# CARGA DE DATOS (igual que antes)
# ============================
def load_data(path, img_size):
    images = []
    masks = []
    num_images = 100  # ajustar si quieres entrenar más rápido
    img_dir = os.path.join(path, "Original")
    mask_dir = os.path.join(path, "Ground truth")
    img_files = sorted(os.listdir(img_dir))[:num_images]
    mask_files = sorted(os.listdir(mask_dir))[:num_images]
    for img_file, mask_file in zip(img_files, mask_files):
        img_path = os.path.join(img_dir, img_file)
        mask_path = os.path.join(mask_dir, mask_file)
        img = load_img(img_path, target_size=img_size)
        mask = load_img(mask_path, target_size=img_size, color_mode="grayscale")
        images.append(img_to_array(img) / 255.0)
        masks.append(img_to_array(mask) / 255.0)
    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)

IMG_SIZE = (512, 512)
x_train, y_train = load_data(TRAIN_PATH, IMG_SIZE)
x_test,  y_test  = load_data(TEST_PATH, IMG_SIZE)

print("Train shapes:", x_train.shape, y_train.shape)
print("Test  shapes:", x_test.shape, y_test.shape)

# ============================
# DATASET TF + AUGMENTACIÓN (igual)
# ============================
def augment_image(image, mask):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    return image, mask

def tf_dataset(x, y, batch_size=8, augment=True):
    dataset = tf.data.Dataset.from_tensor_slices((x, y))
    dataset = dataset.shuffle(buffer_size=len(x))
    if augment:
        dataset = dataset.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

BATCH_SIZE = 8
train_dataset = tf_dataset(x_train, y_train, batch_size=BATCH_SIZE, augment=True)
test_dataset  = tf_dataset(x_test,  y_test,  batch_size=BATCH_SIZE, augment=False)

# ============================
# PÉRDIDAS (mismas que antes)
# ============================
def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return 1.0 - (2.0 * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

bce_loss_fn = tf.keras.losses.BinaryCrossentropy()
def bce_plus_dice_loss(y_true, y_pred):
    return bce_loss_fn(y_true, y_pred) + dice_loss(y_true, y_pred)

def get_loss_fn(cfg="BCE_plus_Dice"):
    if cfg == "BCE":
        return bce_loss_fn
    elif cfg == "Dice":
        return dice_loss
    elif cfg == "BCE_plus_Dice":
        return bce_plus_dice_loss
    else:
        raise ValueError("Unknown loss cfg")

# ============================
# MÉTRICAS: IoU / Dice / Recall (para el entrenamiento)
# ============================
def iou_metric(y_true, y_pred):
    y_pred = tf.round(y_pred)
    y_true = tf.cast(y_true, tf.float32)
    inter = tf.reduce_sum(y_true * y_pred)
    union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) - inter + 1e-7
    return inter / union

def dice_metric(y_true, y_pred):
    y_pred = tf.round(y_pred)
    y_true = tf.cast(y_true, tf.float32)
    inter = tf.reduce_sum(y_true * y_pred)
    return (2. * inter + 1e-7) / (tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + 1e-7)

def recall_metric(y_true, y_pred):
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    y_true = tf.cast(y_true, tf.float32)
    tp = tf.reduce_sum(y_true * y_pred)
    fn = tf.reduce_sum(y_true * (1-y_pred))
    return tp / (tp + fn + 1e-7)

# ============================
# BLOQUES UNET (2xConv estándar)
# ============================
def conv_block(input_tensor, filters, kernel_size=3):
    x = Conv2D(filters, kernel_size, padding="same")(input_tensor)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    x = Conv2D(filters, kernel_size, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    return x

def encoder_block(input_tensor, filters, kernel_size=3):
    x = conv_block(input_tensor, filters, kernel_size)
    p = MaxPooling2D((2, 2))(x)
    return x, p

def decoder_block(input_tensor, skip, filters, kernel_size=3):
    x = Conv2DTranspose(filters, 2, strides=2, padding="same")(input_tensor)
    x = concatenate([x, skip])
    x = conv_block(x, filters, kernel_size)
    return x

# ============================
# BUILD UNET parametrizable por 'depth'
# ============================
def build_unet(depth=4, input_shape=(512,512,3), base_filters=64, kernel_size=3):
    inputs = Input(shape=input_shape)
    skips = []
    x = inputs
    # encoder
    for i in range(depth):
        f = base_filters * (2**i)
        s, x = encoder_block(x, f, kernel_size)
        skips.append(s)
    # bottleneck
    bottleneck_filters = base_filters * (2**depth)
    x = conv_block(x, bottleneck_filters, kernel_size)
    # decoder
    for i in reversed(range(depth)):
        f = base_filters * (2**i)
        x = decoder_block(x, skips[i], f, kernel_size)
    outputs = Conv2D(1, (1,1), activation="sigmoid")(x)
    return Model(inputs, outputs)

# ============================
# CALLBACK para medir tiempo por epoch
# ============================
class EpochTimer(Callback):
    def on_train_begin(self, logs=None):
        self.epoch_times = []
        self._epoch_start = None
    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()
    def on_epoch_end(self, epoch, logs=None):
        self.epoch_times.append(time.time() - self._epoch_start)

# ============================
# CONFIG Y CONSTRUCCIÓN DEL MODELO (usar DEPTH_CFG)
# ============================
LOSS_CFG = "BCE_plus_Dice"   # mantenemos configuración de pérdida
loss_fn = get_loss_fn(LOSS_CFG)

model = build_unet(depth=DEPTH_CFG, input_shape=(512,512,3), base_filters=64, kernel_size=3)
model.compile(optimizer="adam", loss=loss_fn, metrics=["accuracy", iou_metric, dice_metric, recall_metric])
model.summary()

# Guardado/early stop (rutas)
checkpoint_path = f"/kaggle/working/best_model_depth_{DEPTH_CFG}.keras"
checkpoint = ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True, mode='min', verbose=1)
early_stopping = EarlyStopping(monitor='val_loss', patience=15, mode='min', verbose=1, restore_best_weights=True)
epoch_timer = EpochTimer()

# ============================
# ENTRENAMIENTO (EPOCHS = 100 como tu profe pide)
# ============================
EPOCHS = 100
start_total = time.time()
history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=EPOCHS,
    callbacks=[checkpoint, early_stopping, epoch_timer],
    verbose=1
)
end_total = time.time()

# ============================
# MÉTRICAS Y MEDIDAS DE COSTE
# ============================
history_dict = history.history
num_epochs_run = len(history_dict["loss"])
avg_time_per_epoch = np.mean(epoch_timer.epoch_times) if len(epoch_timer.epoch_times)>0 else None
total_time = end_total - start_total
n_params = model.count_params()

# mejor época según val_dice (puedes cambiar por val_loss si prefieres)
if "val_dice_metric" in history_dict:
    best_epoch = int(np.argmax(history_dict["val_dice_metric"])) + 1
    best_val_dice = max(history_dict["val_dice_metric"])
else:
    # fallback a val_loss
    best_epoch = int(np.argmin(history_dict["val_loss"])) + 1
    best_val_dice = None

summary_row = {
    "depth": DEPTH_CFG,
    "n_parameters": n_params,
    "epochs_run": num_epochs_run,
    "avg_time_per_epoch_s": avg_time_per_epoch,
    "total_train_time_s": total_time,
    "best_epoch_by_val_dice": best_epoch,
    "best_val_dice": best_val_dice,
    "final_train_loss": history_dict["loss"][-1],
    "final_val_loss": history_dict["val_loss"][-1],
    "final_train_accuracy": history_dict.get("accuracy", [None])[-1],
    "final_val_accuracy": history_dict.get("val_accuracy", [None])[-1],
    "final_val_iou": history_dict.get("val_iou_metric", [None])[-1],
    "final_val_dice": history_dict.get("val_dice_metric", [None])[-1],
    "final_val_recall": history_dict.get("val_recall_metric", [None])[-1]
}

df_summary = pd.DataFrame([summary_row])
csv_sum_path = f"/kaggle/working/summary_depth_{DEPTH_CFG}.csv"
df_summary.to_csv(csv_sum_path, index=False)
print("Resumen guardado en:", csv_sum_path)
print(df_summary.T)

# ============================
# GUARDAR HISTÓRICO ÉPOCAS EN CSV
# ============================
rows = []
for epoch_idx in range(num_epochs_run):
    rows.append({
        "depth": DEPTH_CFG,
        "epoch": epoch_idx+1,
        "loss": history_dict["loss"][epoch_idx],
        "accuracy": history_dict.get("accuracy",[None])[epoch_idx],
        "val_loss": history_dict["val_loss"][epoch_idx],
        "val_accuracy": history_dict.get("val_accuracy",[None])[epoch_idx],
        "val_iou": history_dict.get("val_iou_metric",[None])[epoch_idx],
        "val_dice": history_dict.get("val_dice_metric",[None])[epoch_idx],
        "val_recall": history_dict.get("val_recall_metric",[None])[epoch_idx],
        "time_epoch_s": epoch_timer.epoch_times[epoch_idx] if epoch_idx < len(epoch_timer.epoch_times) else None
    })
df_history = pd.DataFrame(rows)
history_csv_path = f"/kaggle/working/history_depth_{DEPTH_CFG}.csv"
df_history.to_csv(history_csv_path, index=False)
print("Histórico de épocas guardado en:", history_csv_path)

# ============================
# GRAFICAS: Loss + Accuracy por epoch (misma figura)
# ============================
plt.figure(figsize=(10,6))
plt.plot(df_history["epoch"], df_history["loss"], label="Train Loss", marker='o')
plt.plot(df_history["epoch"], df_history["val_loss"], label="Val Loss", marker='o')
plt.plot(df_history["epoch"], df_history["accuracy"], label="Train Acc", marker='x')
plt.plot(df_history["epoch"], df_history["val_accuracy"], label="Val Acc", marker='x')
plt.title(f"Loss y Accuracy por epoch (depth={DEPTH_CFG})")
plt.xlabel("Epoch")
plt.ylabel("Valor")
plt.legend()
plt.grid(True)
plt.show()

# ============================
# BARRAS: IoU / Dice / Recall (última época de validación)
# ============================
final_iou   = df_history["val_iou"].iloc[-1]   if "val_iou" in df_history else None
final_dice  = df_history["val_dice"].iloc[-1]  if "val_dice" in df_history else None
final_recall= df_history["val_recall"].iloc[-1] if "val_recall" in df_history else None

metrics_names = ["IoU", "Dice", "Recall"]
metrics_vals  = [final_iou, final_dice, final_recall]

plt.figure(figsize=(7,4))
plt.bar(metrics_names, [v if v is not None else 0 for v in metrics_vals])
plt.title(f"Métricas finales (val) - depth={DEPTH_CFG}")
plt.ylim(0,1)
plt.show()

# ============================
# EJEMPLOS VISUALES: mostrar 3 predicciones vs GT (zoom en vasos finos si quieres)
# ============================
n_show = 3
preds = model.predict(x_test[:n_show])
plt.figure(figsize=(12,4*n_show))
for i in range(n_show):
    plt.subplot(n_show,3,i*3+1)
    plt.imshow(x_test[i])
    plt.axis('off')
    if i==0: plt.title("Imagen")

    plt.subplot(n_show,3,i*3+2)
    plt.imshow(y_test[i].squeeze(), cmap='gray')
    plt.axis('off')
    if i==0: plt.title("GT")

    plt.subplot(n_show,3,i*3+3)
    plt.imshow(preds[i].squeeze(), cmap='gray')
    plt.axis('off')
    if i==0: plt.title("Pred")
plt.suptitle(f"Ejemplos (depth={DEPTH_CFG})")
plt.show()
